# Ethereum Phishing Detection — Random Forest baseline versus GCN-embedding hybrid (EPTransNet)

This notebook investigates whether appending GCN-learned graph embeddings to twelve behavioural wallet features improves a Random Forest in classifying phishing-labelled Ethereum wallets. It covers data preparation, the RF1 baseline, a directed two-layer GCN, the RF2 hybrid, validation-based tuning and one final test evaluation.

Experiment design:

- Node = Ethereum wallet/account
- Edge = transaction
- `isp = 1` = phishing-labelled node
- `isp = 0` = not phishing-labelled in this dataset
- Train split: 1:1 positive-to-negative ratio
- Validation split: 1:5
- Test split: 1:20
- A full 1-hop context is retained around all labelled nodes

The two predictors compared are

$$
s^{RF1}_i = RF_1(x_i), \qquad s^{RF2}_i = RF_2([x_i \,\Vert\, h_i]),
$$

where $x_i$ holds the wallet-level behavioural features engineered from the raw transactions and $h_i$ is the frozen GCN embedding of wallet $i$.

| Sections | Content |
|---|---|
| 1–9 | Download, labelled splits, 1-hop subgraph, 12 wallet features, sanity checks (saves CSV files to `/content`) |
| 10–12 | RF1 baseline on the raw features, plots, saved model |
| 13 | Directed sparse GCN, checkpoint selection by validation AP, frozen embeddings |
| 14–16 | RF2, matched tuning of both forests (18 candidates), validation-selected thresholds |
| 17–19 | Frozen protocol, single test evaluation, figures, paired error analysis |
| 20 | Post-hoc analysis of the GCN embeddings: feature correlation, RF2 feature importance, an identity-operator ablation and a one-feature rule baseline (train and validation data only) |
| 21 | Summary and results archive |

**How to run.** Run all cells from the top in Google Colab. Section 2 loads the 1.2 GB `MulDiGraph.pkl` into a NetworkX multigraph, which needs several GB of RAM; use a High-RAM runtime if the session crashes. If `import kagglehub` fails, run `%pip install -q kagglehub`. Sections 13–21 read CSV files saved in `/content` by the earlier sections, and a new Colab runtime does not keep them, so after a restart run the notebook again from the top.

## 1. Imports, reproducibility, and dataset download


In [ ]:
from pathlib import Path
import csv
import gc
import pickle
import random

import numpy as np
import pandas as pd
import kagglehub

SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

dataset_path = Path(
    kagglehub.dataset_download(
        "xblock/ethereum-phishing-transaction-network"
    )
)

print("Dataset path:", dataset_path)

for p in sorted(dataset_path.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"{p.relative_to(dataset_path)} | {size_mb:.2f} MB")


## 2. Read the dataset description and load the full graph

The full NetworkX graph is large, so it is kept in memory only while we:

1. construct the labelled split,
2. extract the 1-hop node set,
3. save the subgraph transactions, and
4. engineer wallet-level features.

After that, the full graph is deleted from RAM.


In [ ]:
readme_path = next(dataset_path.rglob("README.txt"))

readme_text = readme_path.read_text(
    encoding="utf-8",
    errors="replace"
)

# Show only the English section to keep notebook output short.
english_section = readme_text.split("2.中文版")[0]
print(english_section[:8000])

graph_path = next(dataset_path.rglob("MulDiGraph.pkl"))

with graph_path.open("rb") as f:
    G = pickle.load(f)

print("\nGraph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


## 3. Create labelled train / validation / test splits

All 1,165 phishing-labelled nodes are retained.

Nodes with `isp = 0` are sampled from the full graph using **reservoir sampling**, so we do not need to store all ~3 million candidate node IDs in another list.

| Split | Phishing | `isp=0` | Ratio |
|---|---:|---:|---:|
| Train | 815 | 815 | 1:1 |
| Validation | 175 | 875 | 1:5 |
| Test | 175 | 3500 | 1:20 |

The 1:20 test ratio is an experimental imbalance setting. It is **not** claimed to be the true prevalence of phishing accounts on Ethereum.


In [ ]:
# All phishing-labelled nodes.
phishing_nodes = [
    node
    for node, data in G.nodes(data=True)
    if int(data.get("isp", 0)) == 1
]

assert len(phishing_nodes) == 1165, (
    f"Expected 1165 phishing-labelled nodes, "
    f"found {len(phishing_nodes)}"
)

# Split positive nodes.
positives = phishing_nodes.copy()
rng.shuffle(positives)

train_pos = positives[:815]
val_pos   = positives[815:990]
test_pos  = positives[990:]

# Number of isp=0 nodes required for each split.
N_TRAIN_NEG = len(train_pos)
N_VAL_NEG   = len(val_pos) * 5
N_TEST_NEG  = len(test_pos) * 20
TOTAL_NEG   = N_TRAIN_NEG + N_VAL_NEG + N_TEST_NEG

# Reservoir-sample isp=0 nodes.
negative_pool = []
seen_negative = 0

for node, data in G.nodes(data=True):
    if int(data.get("isp", 0)) != 0:
        continue

    seen_negative += 1

    if len(negative_pool) < TOTAL_NEG:
        negative_pool.append(node)
    else:
        j = rng.randrange(seen_negative)
        if j < TOTAL_NEG:
            negative_pool[j] = node

assert len(negative_pool) == TOTAL_NEG

rng.shuffle(negative_pool)

train_neg = negative_pool[:N_TRAIN_NEG]

val_start = N_TRAIN_NEG
val_end = val_start + N_VAL_NEG
val_neg = negative_pool[val_start:val_end]

test_neg = negative_pool[val_end:]

# Labelled node sets.
train_nodes = set(train_pos) | set(train_neg)
val_nodes   = set(val_pos)   | set(val_neg)
test_nodes  = set(test_pos)  | set(test_neg)

# Ensure no labelled node appears in multiple splits.
assert not (train_nodes & val_nodes)
assert not (train_nodes & test_nodes)
assert not (val_nodes & test_nodes)

def make_split(pos_nodes, neg_nodes, split_name):
    return pd.DataFrame({
        "node": list(pos_nodes) + list(neg_nodes),
        "label": [1] * len(pos_nodes) + [0] * len(neg_nodes),
        "split": split_name
    })

split_df = pd.concat([
    make_split(train_pos, train_neg, "train"),
    make_split(val_pos, val_neg, "val"),
    make_split(test_pos, test_neg, "test")
], ignore_index=True)

split_df.to_csv("/content/node_splits.csv", index=False)

print(split_df.groupby(["split", "label"]).size())
print("\nTotal labelled nodes:", len(split_df))


## 4. Build the full 1-hop context node set

Every labelled train/validation/test node is a **seed**.

For each seed, we retain:

- every direct predecessor: wallets that sent to the seed;
- every direct successor: wallets that received from the seed.

These additional nodes provide graph context. They are **not** automatically used as labelled training examples.


In [ ]:
seed_nodes = train_nodes | val_nodes | test_nodes
selected_nodes = set(seed_nodes)

for i, node in enumerate(seed_nodes, 1):
    selected_nodes.update(G.predecessors(node))
    selected_nodes.update(G.successors(node))

    if i % 500 == 0:
        print(
            f"{i}/{len(seed_nodes)} | "
            f"selected nodes = {len(selected_nodes)}"
        )

# Stable order for later files and node-to-index mapping.
selected_node_list = sorted(selected_nodes, key=str)

print("\nLabeled seed nodes:", len(seed_nodes))
print("1-hop nodes total:", len(selected_node_list))
print("Context nodes:", len(selected_node_list) - len(seed_nodes))


## 5. Save the 1-hop subgraph

`subgraph_edges.csv` keeps the raw transaction edges inside the selected node set.

Because the source graph is a `MultiDiGraph`, the same ordered pair of wallets may have many separate transaction edges. We keep those transactions here; they can be collapsed to unique wallet pairs later for the GCN.


In [ ]:
# Store only supervised labels for seed nodes.
label_map = dict(
    zip(split_df["node"], split_df["label"])
)

split_map = dict(
    zip(split_df["node"], split_df["split"])
)

node_file = Path("/content/subgraph_nodes.csv")

with node_file.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["node", "label", "is_seed", "split"])

    for node in selected_node_list:
        is_seed = node in seed_nodes

        writer.writerow([
            node,
            label_map.get(node, ""),
            int(is_seed),
            split_map.get(node, "context")
        ])

edge_file = Path("/content/subgraph_edges.csv")
edge_count = 0

with edge_file.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "source",
        "target",
        "key",
        "amount",
        "timestamp"
    ])

    for u in selected_node_list:
        for _, v, key, data in G.out_edges(
            u,
            keys=True,
            data=True
        ):
            if v not in selected_nodes:
                continue

            writer.writerow([
                u,
                v,
                key,
                data.get("amount"),
                data.get("timestamp")
            ])

            edge_count += 1

            if edge_count % 100000 == 0:
                print("Edges written:", edge_count)

print("\nFinal nodes:", len(selected_node_list))
print("Final transaction edges:", edge_count)
print("Saved:", node_file)
print("Saved:", edge_file)


## 6. Engineer wallet-level behavioural features

The Random Forest cannot consume the graph object directly, so every selected wallet is converted into one tabular row.

Features are calculated from the wallet's transactions in the **full graph**, not only from transactions that remain inside the extracted subgraph.

The baseline features are:

- incoming and outgoing transaction counts;
- unique senders and receivers;
- total incoming and outgoing amount;
- mean incoming and outgoing amount;
- maximum incoming and outgoing amount;
- activity duration;
- net incoming minus outgoing amount.

`total_transactions` and `total_amount` are not included in the final model feature list because they are exact sums of features already present.


In [ ]:
def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0

def to_timestamp(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


rows = []
n_selected = len(selected_node_list)

for idx, node in enumerate(selected_node_list, 1):
    num_in = 0
    num_out = 0

    total_in = 0.0
    total_out = 0.0

    max_in = 0.0
    max_out = 0.0

    senders = set()
    receivers = set()

    first_time = None
    last_time = None

    # Incoming transactions: u -> node
    for u, _, _, data in G.in_edges(
        node,
        keys=True,
        data=True
    ):
        amount = to_float(data.get("amount"))
        timestamp = to_timestamp(data.get("timestamp"))

        num_in += 1
        total_in += amount
        max_in = max(max_in, amount)
        senders.add(u)

        if timestamp is not None:
            if first_time is None or timestamp < first_time:
                first_time = timestamp
            if last_time is None or timestamp > last_time:
                last_time = timestamp

    # Outgoing transactions: node -> v
    for _, v, _, data in G.out_edges(
        node,
        keys=True,
        data=True
    ):
        amount = to_float(data.get("amount"))
        timestamp = to_timestamp(data.get("timestamp"))

        num_out += 1
        total_out += amount
        max_out = max(max_out, amount)
        receivers.add(v)

        if timestamp is not None:
            if first_time is None or timestamp < first_time:
                first_time = timestamp
            if last_time is None or timestamp > last_time:
                last_time = timestamp

    rows.append({
        "node": node,
        "num_in": num_in,
        "num_out": num_out,
        "unique_senders": len(senders),
        "unique_receivers": len(receivers),
        "total_in": total_in,
        "total_out": total_out,
        "mean_in": total_in / num_in if num_in else 0.0,
        "mean_out": total_out / num_out if num_out else 0.0,
        "max_in": max_in,
        "max_out": max_out,
        "activity_duration": (
            last_time - first_time
            if first_time is not None and last_time is not None
            else 0.0
        ),
        "net_amount": total_in - total_out
    })

    if idx % 2000 == 0:
        print(f"{idx} / {n_selected}")

features = pd.DataFrame(rows)

print("\nFeature table shape:", features.shape)
display(features.head())


## 7. Attach labels and split information

Only train/validation/test seed nodes have supervised labels. Context nodes remain unlabelled for the classifier.


In [ ]:
features = features.merge(
    split_df,
    on="node",
    how="left"
)

features["split"] = features["split"].fillna("context")

print(features["split"].value_counts())

print("\nLabelled class counts:")
print(
    features[
        features["split"] != "context"
    ]
    .groupby(["split", "label"])
    .size()
)

features.to_csv(
    "/content/node_features_raw.csv",
    index=False
)

print("\nSaved: /content/node_features_raw.csv")


## 8. Sanity-check the engineered features

The transaction features are expected to be strongly right-skewed. This is not a problem for the Random Forest baseline because decision trees do not require feature standardisation.

For the later GCN, a separate transformed/scaled representation can be created using statistics fitted only on the training split.


In [ ]:
model_features = [
    "num_in",
    "num_out",
    "unique_senders",
    "unique_receivers",
    "total_in",
    "total_out",
    "mean_in",
    "mean_out",
    "max_in",
    "max_out",
    "activity_duration",
    "net_amount"
]

# Basic validity checks.
assert features[model_features].isna().sum().sum() == 0
assert np.isfinite(
    features[model_features].to_numpy(dtype=float)
).all()

display(
    features[model_features]
    .describe()
    .T
)


## 9. Release the full graph from RAM

From this point onward, the Random Forest uses the saved tabular feature table. The original ~3-million-node NetworkX graph is no longer required for the baseline.


In [ ]:
del G
gc.collect()

print("Full graph deleted from RAM.")


## 10. Prepare the Random Forest inputs

The baseline classifier uses only the **labelled seed nodes**.

No standardisation is applied because Random Forests split features by thresholds and do not require a common numerical scale.

At the end of this section, `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, and `y_test` are ready.


In [ ]:
train_df = features[features["split"] == "train"].copy()
val_df   = features[features["split"] == "val"].copy()
test_df  = features[features["split"] == "test"].copy()

X_train = train_df[model_features]
y_train = train_df["label"].astype(int)

X_val = val_df[model_features]
y_val = val_df["label"].astype(int)

X_test = test_df[model_features]
y_test = test_df["label"].astype(int)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "| y_test: ", y_test.shape)

print("\nTrain label counts:")
print(y_train.value_counts().sort_index())

print("\nValidation label counts:")
print(y_val.value_counts().sort_index())

print("\nTest label counts:")
print(y_test.value_counts().sort_index())


## 11. Random Forest baseline (RF1)

RF1 is a 300-tree Random Forest on the 12 raw features (`max_features="sqrt"`, seed 42, no feature scaling). Hyper-parameter tuning uses the **validation set** only (Section 15); the test set is reserved for the final evaluation (Section 18).

The cell below fits RF1 on the training wallets, and the next cell reports its validation performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1
)

print("Random Forest is ready.")
print("Next command:")
print("rf.fit(X_train, y_train)")

forbidden_cols = {"node", "label", "split", "isp", "is_seed"}
assert not forbidden_cols.intersection(X_train.columns)

assert len(X_train) == len(y_train)
assert set(y_train.unique()) == {0, 1}

rf.fit(X_train, y_train)

print("Training completed.")
print("Training samples:", len(X_train))
print("Features:", X_train.shape[1])
print("Number of trees:", len(rf.estimators_))

In [ ]:
from sklearn import metrics

positive_col = list(rf.classes_).index(1)

val_scores = rf.predict_proba(X_val)[:, positive_col]

threshold = 0.5
val_predictions = (val_scores >= threshold).astype(int)

print("RANDOM FOREST — VALIDATION")
print(f"Threshold: {threshold}\n")

print(
    metrics.classification_report(
        y_val,
        val_predictions,
        labels=[0, 1],
        target_names=["isp=0", "phishing-labelled"],
        digits=4,
        zero_division=0
    )
)

val_ap = metrics.average_precision_score(y_val, val_scores)
print(f"Average Precision: {val_ap:.4f}")

cm = metrics.confusion_matrix(
    y_val,
    val_predictions,
    labels=[0, 1]
)

display(
    pd.DataFrame(
        cm,
        index=["Actual isp=0", "Actual phishing"],
        columns=["Predicted isp=0", "Predicted phishing"]
    )
)

In [ ]:
figure_dir = Path("/content/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

def evaluate_rf(model, X, y, split_name, threshold=0.5):
    y_true = np.asarray(y, dtype=int)

    positive_col = list(model.classes_).index(1)
    scores = model.predict_proba(X)[:, positive_col]
    predictions = (scores >= threshold).astype(int)

    cm = metrics.confusion_matrix(
        y_true, predictions, labels=[0, 1]
    )
    tn, fp, fn, tp = cm.ravel()

    ap = metrics.average_precision_score(y_true, scores)
    prevalence = np.mean(y_true == 1)

    result = {
        "Split": split_name,
        "Samples": len(y_true),
        "Positive proportion": prevalence,
        "Threshold": threshold,
        "Precision": metrics.precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": metrics.recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1 phishing": metrics.f1_score(
            y_true, predictions, zero_division=0
        ),
        "Macro-F1": metrics.f1_score(
            y_true, predictions, average="macro", zero_division=0
        ),
        "AP": ap,
        "False positive rate": fp / (fp + tn) if fp + tn else np.nan,
    }

    # Confusion matrix
    plt.figure(figsize=(6, 5))

    metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["isp=0", "Phishing-labelled"]
    ).plot(
        ax=plt.gca(),
        values_format="d",
        colorbar=False
    )

    plt.title(f"RF — {split_name}\nThreshold = {threshold}")
    plt.tight_layout()
    plt.savefig(
        figure_dir / f"rf_{split_name.lower()}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    # Precision–Recall curve
    precision, recall, _ = metrics.precision_recall_curve(
        y_true, scores
    )

    plt.figure(figsize=(6, 5))
    plt.step(
        recall, precision,
        where="post",
        label=f"RF — AP = {ap:.4f}"
    )
    plt.axhline(
        prevalence,
        linestyle="--",
        label=f"Positive proportion = {prevalence:.3f}"
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"RF — {split_name} Precision–Recall Curve")
    plt.xlim(0, 1)
    plt.ylim(0, 1.05)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        figure_dir / f"rf_{split_name.lower()}_precision_recall.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    return result

## 12. RF1 plots and saved baseline

The helper `evaluate_rf(...)` defined in the previous cell is called here on the **train and validation** splits only (threshold 0.5) to produce metrics, confusion matrices and a precision–recall plot. The fitted RF1 is saved to `/content/rf_outputs` so that later sections can reuse it without refitting. No fitting, tuning, sampling or test-set evaluation happens in this cell.

**Running the notebook.** The GCN section reads `node_features_raw.csv`, `node_splits.csv` and `subgraph_edges.csv` from `/content`; they are produced by Sections 3–8 and are not stored inside this notebook. A new Colab runtime does not keep them, so after a restart run the notebook again from the top.

In [ ]:
# Use the already fitted RF1. Do not call fit() here.
rf1_required = [
    "rf", "evaluate_rf", "X_train", "y_train", "X_val", "y_val", "val_df"
]
rf1_missing = [name for name in rf1_required if name not in globals()]
if rf1_missing:
    raise RuntimeError(
        "Run/restore the original RF cells first. Missing: " + ", ".join(rf1_missing)
    )
if not hasattr(rf, "estimators_"):
    raise RuntimeError("RF1 has not been fitted. Run its original training cell first.")

rf_baseline = rf
RF1_THRESHOLD = 0.5
rf1_output_dir = Path("/content/rf_outputs")
rf1_output_dir.mkdir(parents=True, exist_ok=True)

rf1_train_result = evaluate_rf(
    rf_baseline, X_train, y_train, "Train", RF1_THRESHOLD
)
rf1_val_result = evaluate_rf(
    rf_baseline, X_val, y_val, "Validation", RF1_THRESHOLD
)
rf1_summary = pd.DataFrame([rf1_train_result, rf1_val_result]).set_index("Split")
display(rf1_summary.round(4))
rf1_summary.to_csv(rf1_output_dir / "rf_train_validation_metrics.csv")

with (rf1_output_dir / "rf_baseline.pkl").open("wb") as rf1_handle:
    pickle.dump(
        {
            "model": rf_baseline,
            "feature_columns": list(X_train.columns),
            "threshold": RF1_THRESHOLD,
            "seed": SEED,
        },
        rf1_handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

rf1_positive_col = list(rf_baseline.classes_).index(1)
rf1_val_scores = rf_baseline.predict_proba(X_val)[:, rf1_positive_col]
pd.DataFrame({
    "node": val_df["node"].to_numpy(),
    "label": np.asarray(y_val, dtype=int),
    "score": rf1_val_scores,
    "prediction": (rf1_val_scores >= RF1_THRESHOLD).astype(int),
}).to_csv(rf1_output_dir / "rf_validation_predictions.csv", index=False)

print("RF1 was not retrained. Baseline saved to:", rf1_output_dir)
print("RF figures saved by evaluate_rf to:", figure_dir)
print("No test metrics were computed. Only reload pickle files you trust.")

## 13. GCN training and embeddings for RF2

This section trains a two-layer directed GCN with an auxiliary neural classification head, selects its checkpoint by validation Average Precision (AP), plots training diagnostics, and exports 32-dimensional embeddings keyed by wallet address. It reuses the files saved by the earlier sections and does not repeat sampling or feature engineering. RF1 is not modified; RF2 training and the test-set evaluation come later (Sections 14–18).

Inputs, all read from `/content`:

- `node_features_raw.csv`
- `node_splits.csv`
- `subgraph_edges.csv`

This section never reloads `MulDiGraph.pkl` or resamples the splits.

### Experimental scope

This is a retrospective, transductive benchmark on the already selected graph. Features/edges of train, validation, test and context nodes are available to message passing; only training labels produce optimization gradients. Validation labels select the checkpoint. Test labels are not used for fitting, model selection or metric computation.

The split is train 815/815, validation 175/875 and test 175/3500 (phishing-labelled/`isp=0`). These ratios are experimental, not estimates of Ethereum prevalence. `isp=0` means not phishing-labelled in this release, not independently verified safe. Sampling from a phishing-seeded crawl and using retrospective transaction summaries limit real-world generalization claims.

The GCN implementation and hyper-parameters below are experimental choices, not a reproduction of FraudGCN.

### 13.1 Setup — GCN imports

Uses PyTorch's sparse matrix multiplication directly for graph convolution. No NetworkX graph, PyG or DGL is loaded in this stage. Reuse Path, random, NumPy, pandas, matplotlib and metrics from the RF section. This cell adds PyTorch, JSON/time utilities and StandardScaler. The commented install command is only needed if an import fails.

The random seed is fixed, but exact numerical reproducibility is not guaranteed across hardware/library versions, especially GPU kernels.

In [ ]:
# Run only if a package is missing:
# %pip install -q torch numpy pandas scikit-learn matplotlib

# Path, random, np, pd, plt and metrics are imported in the RF section.
import json
import time

import torch
from torch import nn
from sklearn.preprocessing import StandardScaler

GCN_SEED = 42
GCN_DATA_DIR = Path("/content")
GCN_OUT = GCN_DATA_DIR / "gcn_outputs"
GCN_OUT.mkdir(parents=True, exist_ok=True)
(GCN_OUT / "figures").mkdir(exist_ok=True)

random.seed(GCN_SEED)
np.random.seed(GCN_SEED)
torch.manual_seed(GCN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GCN_SEED)

gcn_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", gcn_device)
print("PyTorch:", torch.__version__)
print("Inputs:", GCN_DATA_DIR)
print("Outputs:", GCN_OUT)

### 13.2 Reload the saved features and preserve the existing split

The feature table is not regenerated. Join the authoritative split file by `node`, not by row number. Sort wallet IDs once and use that order for features, edges and embeddings.

Only the 12 named behavioral features enter the network. Wallet ID, label, split and seed membership are excluded. Context labels stay unavailable. This cell checks the previously agreed split counts; it does not choose a new split.

In [ ]:
GCN_FEATURE_COLS = [
    "num_in", "num_out", "unique_senders", "unique_receivers",
    "total_in", "total_out", "mean_in", "mean_out",
    "max_in", "max_out", "activity_duration", "net_amount",
]

for name in ["node_features_raw.csv", "node_splits.csv", "subgraph_edges.csv"]:
    if not (GCN_DATA_DIR / name).is_file():
        raise FileNotFoundError(
            f"Missing {GCN_DATA_DIR / name}. Restore the saved CSV; "
            "do not re-download the large graph just for this stage."
        )

gcn_raw = pd.read_csv(GCN_DATA_DIR / "node_features_raw.csv", dtype={"node": str})
gcn_splits = pd.read_csv(GCN_DATA_DIR / "node_splits.csv", dtype={"node": str})

required = {"node", *GCN_FEATURE_COLS}
if not required.issubset(gcn_raw.columns):
    raise ValueError(f"Missing feature columns: {required - set(gcn_raw.columns)}")
if not {"node", "label", "split"}.issubset(gcn_splits.columns):
    raise ValueError("node_splits.csv must contain node, label and split.")

for name, frame in [("features", gcn_raw), ("splits", gcn_splits)]:
    if frame["node"].isna().any() or frame["node"].duplicated().any():
        raise ValueError(f"Missing or duplicate wallet IDs in {name}.")
    if frame["node"].str.strip().ne(frame["node"]).any():
        raise ValueError(f"Whitespace in {name} wallet IDs: inspect rather than silently rename.")

if not gcn_splits["split"].isin(["train", "val", "test"]).all():
    raise ValueError("Unexpected split name; expected train/val/test.")
if not gcn_splits["label"].isin([0, 1]).all():
    raise ValueError("Supervised labels must be 0 or 1.")
if not gcn_splits["node"].isin(gcn_raw["node"]).all():
    raise ValueError("Some labelled wallets have no feature row.")

# The raw feature CSV may also store metadata. Verify that it agrees.
for column in ["label", "split"]:
    if column in gcn_raw.columns:
        saved = gcn_raw.set_index("node")[column].reindex(gcn_splits["node"])
        reference = gcn_splits[column].to_numpy()
        if saved.isna().any() or not np.all(saved.to_numpy() == reference):
            raise ValueError(f"Conflicting {column} metadata between the two CSVs.")

gcn_nodes = (
    gcn_raw[["node"] + GCN_FEATURE_COLS]
    .merge(gcn_splits[["node", "label", "split"]], on="node",
           how="left", validate="one_to_one")
    .sort_values("node", kind="stable")
    .reset_index(drop=True)
)
gcn_nodes["split"] = gcn_nodes["split"].fillna("context")
gcn_nodes[GCN_FEATURE_COLS] = gcn_nodes[GCN_FEATURE_COLS].apply(
    pd.to_numeric, errors="raise"
)

gcn_train_np = gcn_nodes["split"].eq("train").to_numpy()
gcn_val_np = gcn_nodes["split"].eq("val").to_numpy()
gcn_test_np = gcn_nodes["split"].eq("test").to_numpy()

gcn_counts = gcn_splits.groupby(["split", "label"]).size()
expected_counts = {
    ("train", 0): 815, ("train", 1): 815,
    ("val", 0): 875, ("val", 1): 175,
    ("test", 0): 3500, ("test", 1): 175,
}
if gcn_counts.to_dict() != expected_counts:
    raise ValueError("Split counts differ from the agreed design. Inspect node_splits.csv.")

gcn_node_ids = gcn_nodes["node"].to_numpy(dtype=str)
gcn_node_to_idx = {gcn_node: gcn_i for gcn_i, gcn_node in enumerate(gcn_node_ids)}
GCN_N = len(gcn_nodes)

pd.DataFrame({
    "node_index": np.arange(GCN_N),
    "node": gcn_node_ids,
    "split": gcn_nodes["split"],
}).to_csv(GCN_OUT / "node_mapping.csv", index=False)

print(gcn_counts)
print("All graph nodes:", GCN_N)
print("Context nodes:", int((~(gcn_train_np | gcn_val_np | gcn_test_np)).sum()))
print("Input features:", len(GCN_FEATURE_COLS))

### 13.3 Create a separate, scaled input for the GCN

Keep RF1's original 12-feature representation unchanged. For GCN only, use `log1p` on non-negative features and a signed log on `net_amount`:

$$T(x)=\log(1+x),\qquad T_{net}(x)=\operatorname{sign}(x)\log(1+|x|).$$

Then fit `StandardScaler` **only on training nodes** and transform every node with those fixed parameters. This is an initial numerical preprocessing choice, not a guarantee of improved predictions.

Reject missing/infinite values instead of silently filling them here. Finite input checks cannot reveal whether an upstream extraction step already replaced a malformed value by zero; that limitation remains part of the data audit. Amounts and durations remain in their released dataset units.

In [ ]:
gcn_raw_array = gcn_nodes[GCN_FEATURE_COLS].to_numpy(dtype=np.float64)
if not np.isfinite(gcn_raw_array).all():
    raise ValueError("Non-finite raw features: fix the feature audit before training.")

gcn_net_index = GCN_FEATURE_COLS.index("net_amount")
gcn_nonnegative = [gcn_i for gcn_i in range(len(GCN_FEATURE_COLS)) if gcn_i != gcn_net_index]
if (gcn_raw_array[:, gcn_nonnegative] < 0).any():
    raise ValueError("Unexpected negative count, amount or duration. Inspect the source.")

gcn_log_array = gcn_raw_array.copy()
gcn_log_array[:, gcn_nonnegative] = np.log1p(gcn_log_array[:, gcn_nonnegative])
net = gcn_log_array[:, gcn_net_index].copy()
gcn_log_array[:, gcn_net_index] = np.sign(net) * np.log1p(np.abs(net))

gcn_scaler = StandardScaler()
gcn_scaler.fit(gcn_log_array[gcn_train_np])
gcn_scaled_array = gcn_scaler.transform(gcn_log_array).astype(np.float32)
if not np.isfinite(gcn_scaled_array).all():
    raise ValueError("GCN preprocessing produced non-finite values.")

# Save transparent preprocessing parameters, not a large graph pickle.
preprocessing_info = {
    "feature_columns": GCN_FEATURE_COLS,
    "transform": "log1p on non-negative columns; signed log1p on net_amount",
    "scaler_fit_split": "train",
    "mean": gcn_scaler.mean_.tolist(),
    "scale": gcn_scaler.scale_.tolist(),
}
(GCN_OUT / "preprocessing.json").write_text(
    json.dumps(preprocessing_info, indent=2), encoding="utf-8"
)
np.save(GCN_OUT / "node_features_scaled.npy", gcn_scaled_array)

gcn_x = torch.from_numpy(gcn_scaled_array).to(gcn_device)
gcn_train_idx = torch.from_numpy(np.flatnonzero(gcn_train_np)).to(gcn_device)
gcn_val_idx = torch.from_numpy(np.flatnonzero(gcn_val_np)).to(gcn_device)

# Only train and validation targets are constructed as tensors.
# Test/context labels do not enter loss or checkpoint selection.
gcn_y_train_np = gcn_nodes.loc[gcn_train_np, "label"].to_numpy(dtype=np.int64)
gcn_y_val_np = gcn_nodes.loc[gcn_val_np, "label"].to_numpy(dtype=np.int64)
gcn_y_train = torch.from_numpy(gcn_y_train_np).to(gcn_device)
gcn_y_val = torch.from_numpy(gcn_y_val_np).to(gcn_device)

print("X for GCN:", tuple(gcn_x.shape))
print("Scaler fitted on", gcn_scaler.n_samples_seen_, "training nodes.")

### 13.4 Build the sparse graph operator — keep transaction direction

This stage **preserves the earlier directed-edge choice**. Collapse repeated `(source, target)` transactions to one binary structural edge. `A -> B` and `B -> A` remain different edges; no reverse edge is invented. Transaction amount/frequency remain represented in the behavioral node summaries, but this does not retain every pair-specific statistic in the structural graph.

For an edge `j -> i`, the sparse matrix has an entry at **row i, column j**, so multiplication aggregates sender features into the receiver. With self-loops:

$$B_{ij}=1\text{ if } j\to i,\quad \widetilde B=B+I,$$
$$d_i=\sum_j\widetilde B_{ij},\quad P_{ij}=\frac{\widetilde B_{ij}}{\sqrt{d_i d_j}}.$$

Self-transactions are removed from the structural edge list before one self-loop is inserted per node. Their transaction statistics have already been retained in raw node features.

**This is a directed use of GCN normalization, with incoming degrees**, matching the node-wise source-to-target convention documented for `GCNConv`. The original symmetric/undirected spectral interpretation should not be claimed for this directed matrix. A node only receives direct messages from predecessors and itself; outgoing behavior is still in its feature vector. Two layers propagate at most two steps **inside the retained induced graph**, not a complete two-hop neighborhood of the original network.

Use sparse COO storage. Never allocate a dense `N x N` array. CSVs are read in chunks, retaining only two endpoint columns.

In [ ]:
def read_directed_pairs(csv_path, node_to_idx, chunksize=200_000):
    """Map saved transaction endpoints to node indices; deduplicate directed pairs."""
    pieces = []
    transaction_count = 0
    for chunk in pd.read_csv(
        csv_path, usecols=["source", "target"],
        dtype={"source": str, "target": str}, chunksize=chunksize
    ):
        transaction_count += len(chunk)
        src = chunk["source"].map(node_to_idx)
        dst = chunk["target"].map(node_to_idx)
        if src.isna().any() or dst.isna().any():
            raise ValueError("An edge endpoint is missing from node_features_raw.csv.")
        pairs = np.column_stack([
            src.to_numpy(dtype=np.int64), dst.to_numpy(dtype=np.int64)
        ])
        pieces.append(np.unique(pairs, axis=0))
    if not pieces:
        raise ValueError("Empty transaction edge file.")
    return np.unique(np.concatenate(pieces, axis=0), axis=0), transaction_count


def normalized_incoming_graph(pairs, n_nodes):
    """Sparse P[i,j] for message j->i, with one self-loop and incoming-degree normalization."""
    nonself = pairs[pairs[:, 0] != pairs[:, 1]]
    own = np.arange(n_nodes, dtype=np.int64)
    src = np.concatenate([nonself[:, 0], own])
    dst = np.concatenate([nonself[:, 1], own])
    degree = np.bincount(dst, minlength=n_nodes).astype(np.float64)
    values = (1.0 / np.sqrt(degree[dst] * degree[src])).astype(np.float32)
    indices = torch.from_numpy(np.vstack([dst, src]))
    return torch.sparse_coo_tensor(
        indices, torch.from_numpy(values), (n_nodes, n_nodes)
    ).coalesce()


gcn_pairs, gcn_transaction_count = read_directed_pairs(
    GCN_DATA_DIR / "subgraph_edges.csv", gcn_node_to_idx
)
np.save(GCN_OUT / "directed_edge_pairs.npy", gcn_pairs)
gcn_self_pairs = int((gcn_pairs[:, 0] == gcn_pairs[:, 1]).sum())
gcn_adj = normalized_incoming_graph(gcn_pairs, GCN_N).to(gcn_device)

print("Transaction rows:", gcn_transaction_count)
print("Unique directed pairs:", len(gcn_pairs))
print("Existing self-pairs:", gcn_self_pairs)
print("Nonzeros after adding one self-loop per node:", gcn_adj._nnz())
print("Sparse adjacency shape:", tuple(gcn_adj.shape))

### 13.5 Define the GCN and its temporary classification head

Each convolution calculates:

$$H^{(l+1)}=\operatorname{ReLU}(P H^{(l)}W_l+b_l).$$

The implementation multiplies sparse `P` by dense features using `torch.sparse.mm`; it does not treat the adjacency as trainable weights.

**Architecture:** 12 input features -> GCN 64 -> ReLU -> dropout 0.3 -> GCN 32 -> ReLU -> 32-dimensional embedding -> Linear 2 logits.

The linear head is necessary to supervise the representation with cross-entropy. RF2 will later use the embedding **before this head**, not just a binary prediction. Dimensions and hyperparameters are starting choices, not tuned settings.

In [ ]:
class GraphConvolution(nn.Module):
    """Compute P @ X @ W + b using sparse-dense multiplication."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adjacency):
        return torch.sparse.mm(adjacency, x @ self.weight) + self.bias


class WalletGCN(nn.Module):
    def __init__(self, in_dim=12, hidden_dim=64, embedding_dim=32, dropout=0.3):
        super().__init__()
        self.conv1 = GraphConvolution(in_dim, hidden_dim)
        self.conv2 = GraphConvolution(hidden_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(embedding_dim, 2)

    def encode(self, x, adjacency):
        h = torch.relu(self.conv1(x, adjacency))
        h = self.dropout(h)
        return torch.relu(self.conv2(h, adjacency))

    def forward(self, x, adjacency):
        embeddings = self.encode(x, adjacency)
        return self.head(embeddings), embeddings


gcn_config = {
    "in_dim": len(GCN_FEATURE_COLS),
    "hidden_dim": 64,
    "embedding_dim": 32,
    "dropout": 0.3,
}
torch.manual_seed(GCN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GCN_SEED)
gcn = WalletGCN(**gcn_config).to(gcn_device)
print(gcn)
print("Trainable parameters:", sum(p.numel() for p in gcn.parameters()))

# Forward sanity check without labels.
gcn.eval()
with torch.no_grad():
    check_logits, check_h = gcn(gcn_x, gcn_adj)
assert check_logits.shape == (GCN_N, 2)
assert check_h.shape == (GCN_N, gcn_config["embedding_dim"])
assert torch.isfinite(check_logits).all() and torch.isfinite(check_h).all()
print("Logits:", tuple(check_logits.shape), "| Embeddings:", tuple(check_h.shape))
del check_logits, check_h

### 13.6 Train — train gradients, validation checkpoint selection, no test metrics

Unweighted cross-entropy is the starting objective because the supervised training split is 1:1. It consumes **raw logits**, not pre-softmaxed probabilities:

$$L=-\frac{1}{|T|}\sum_{i\in T}\log\operatorname{softmax}(z_i)_{y_i}.$$

Adam: learning rate `0.001`, weight decay `0.0005`; up to 200 full-graph epochs; early stopping after 30 epochs without validation AP improvement. Graph context participates in forward propagation, but only 1,630 supervised train nodes enter the loss used for `backward()`.

After each update, compute train/validation losses in `eval()` mode with dropout disabled. These are the comparable data-loss curves; weight decay is not included in the plotted CE. The optimization loss with dropout is also logged separately. Restore the checkpoint with the highest validation AP, not the final epoch by default.

Train and validation have different class ratios, so AP/F1 and loss gaps are not pure estimates of overfitting. No test curve is generated during training.

In [ ]:
GCN_MAX_EPOCHS = 200
GCN_PATIENCE = 30
GCN_LR = 0.001
GCN_WEIGHT_DECAY = 0.0005

gcn_criterion = nn.CrossEntropyLoss()
gcn_optimizer = torch.optim.Adam(
    gcn.parameters(), lr=GCN_LR, weight_decay=GCN_WEIGHT_DECAY
)

gcn_history_rows = []
gcn_best_ap = -float("inf")
gcn_best_epoch = 0
gcn_best_state = None
gcn_stale = 0
gcn_start = time.perf_counter()

for epoch in range(1, GCN_MAX_EPOCHS + 1):
    gcn.train()
    gcn_optimizer.zero_grad(set_to_none=True)
    logits, gcn_unused = gcn(gcn_x, gcn_adj)
    optimization_loss = gcn_criterion(logits[gcn_train_idx], gcn_y_train)
    if not torch.isfinite(optimization_loss):
        raise FloatingPointError("Non-finite GCN loss; inspect features and normalization.")
    optimization_loss.backward()
    gcn_optimizer.step()
    optimization_value = float(optimization_loss.item())
    del logits, optimization_loss, gcn_unused

    gcn.eval()
    with torch.no_grad():
        evaluated_logits, gcn_unused = gcn(gcn_x, gcn_adj)
        if not torch.isfinite(evaluated_logits).all():
            raise FloatingPointError("Non-finite logits after optimizer step.")
        train_ce = float(gcn_criterion(evaluated_logits[gcn_train_idx], gcn_y_train).item())
        val_ce = float(gcn_criterion(evaluated_logits[gcn_val_idx], gcn_y_val).item())
        train_scores = evaluated_logits[gcn_train_idx].softmax(dim=1)[:, 1].cpu().numpy()
        val_scores_gcn_epoch = evaluated_logits[gcn_val_idx].softmax(dim=1)[:, 1].cpu().numpy()

    train_ap = float(metrics.average_precision_score(gcn_y_train_np, train_scores))
    gcn_val_ap_epoch = float(metrics.average_precision_score(gcn_y_val_np, val_scores_gcn_epoch))
    gcn_history_rows.append({
        "epoch": epoch,
        "optimization_loss_dropout_on": optimization_value,
        "train_loss": train_ce,
        "val_loss": val_ce,
        "train_ap": train_ap,
        "val_ap": gcn_val_ap_epoch,
    })
    pd.DataFrame(gcn_history_rows).to_csv(GCN_OUT / "gcn_history.csv", index=False)

    if gcn_val_ap_epoch > gcn_best_ap + 1e-6:
        gcn_best_ap = gcn_val_ap_epoch
        gcn_best_epoch = epoch
        gcn_stale = 0
        gcn_best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in gcn.state_dict().items()
        }
        torch.save(gcn_best_state, GCN_OUT / "gcn_best_state.pt")
    else:
        gcn_stale += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train CE {train_ce:.4f} | val CE {val_ce:.4f} "
              f"| train AP {train_ap:.4f} | val AP {gcn_val_ap_epoch:.4f}")
    del evaluated_logits, gcn_unused
    if gcn_stale >= GCN_PATIENCE:
        print("Early stopping at epoch", epoch)
        break

if gcn_best_state is None:
    raise RuntimeError("No valid checkpoint was selected.")
gcn.load_state_dict(gcn_best_state)
gcn.eval()
gcn_history = pd.DataFrame(gcn_history_rows)

run_info = {
    "model": gcn_config,
    "seed": GCN_SEED,
    "learning_rate": GCN_LR,
    "weight_decay": GCN_WEIGHT_DECAY,
    "max_epochs": GCN_MAX_EPOCHS,
    "patience": GCN_PATIENCE,
    "selection_metric": "validation_average_precision",
    "best_epoch": gcn_best_epoch,
    "best_validation_ap": gcn_best_ap,
    "graph_mode": "unique directed source-to-target edges; incoming-degree normalization",
    "self_loops": "exactly one per node",
    "n_nodes": GCN_N,
    "n_unique_directed_pairs": int(len(gcn_pairs)),
    "n_train": int(gcn_train_np.sum()),
    "n_val": int(gcn_val_np.sum()),
    "torch_version": str(torch.__version__),
    "device": str(gcn_device),
    "elapsed_seconds": time.perf_counter() - gcn_start,
    "test_metrics_computed": False,
}
(GCN_OUT / "gcn_config.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")
print("Best epoch:", gcn_best_epoch)
print(f"Best validation AP: {gcn_best_ap:.4f}")
print("Checkpoint restored. Test set has not been scored.")

### 13.7 Plot learning curves

Separate figures show CE loss and AP over epochs. Train/validation CE is measured with dropout disabled after the same epoch update. The vertical line identifies the selected checkpoint. The two splits have different class proportions, so do not attribute every gap to overfitting.

In [ ]:
def save_gcn_figure(filename):
    plt.tight_layout()
    plt.savefig(GCN_OUT / "figures" / filename, dpi=250, bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(7, 4))
plt.plot(gcn_history["epoch"], gcn_history["train_loss"], label="Train CE (eval mode)")
plt.plot(gcn_history["epoch"], gcn_history["val_loss"], label="Validation CE (eval mode)")
plt.axvline(gcn_best_epoch, linestyle="--", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("GCN — Training and validation loss")
plt.legend()
plt.grid(alpha=0.25)
save_gcn_figure("gcn_loss.png")

plt.figure(figsize=(7, 4))
plt.plot(gcn_history["epoch"], gcn_history["train_ap"], label="Train AP (1:1)")
plt.plot(gcn_history["epoch"], gcn_history["val_ap"], label="Validation AP (1:5)")
plt.axvline(gcn_best_epoch, linestyle="--", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Average Precision")
plt.ylim(0, 1.05)
plt.title("GCN — AP by epoch (different split class ratios)")
plt.legend()
plt.grid(alpha=0.25)
save_gcn_figure("gcn_ap.png")

### 13.8 Diagnose the auxiliary GCN classifier on train and validation

The fixed initial threshold is 0.5. Report phishing-class precision/recall/F1, macro-F1, AP and false-positive rate, with confusion matrices and Precision–Recall curves. AP is calculated from scores, not hard labels.

These are diagnostics of the GCN's temporary head, **not the final GCN+RF result**. Sampling a balanced training set also means these scores should not be claimed to be calibrated real-world phishing probabilities.

In [ ]:
gcn.eval()
with torch.no_grad():
    gcn_final_logits, gcn_h_tensor = gcn(gcn_x, gcn_adj)
    gcn_all_scores = gcn_final_logits.softmax(dim=1)[:, 1].cpu().numpy()

gcn_metric_rows = []
for split_name, selected_mask, labels in [
    ("train", gcn_train_np, gcn_y_train_np),
    ("val", gcn_val_np, gcn_y_val_np),
]:
    scores = gcn_all_scores[selected_mask]
    predictions = (scores >= 0.5).astype(np.int64)
    gcn_cm = metrics.confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = gcn_cm.ravel()
    ap = float(metrics.average_precision_score(labels, scores))
    gcn_metric_rows.append({
        "split": split_name, "samples": len(labels), "threshold": 0.5,
        "precision": metrics.precision_score(labels, predictions, zero_division=0),
        "recall": metrics.recall_score(labels, predictions, zero_division=0),
        "f1_phishing": metrics.f1_score(labels, predictions, zero_division=0),
        "macro_f1": metrics.f1_score(labels, predictions, average="macro", zero_division=0),
        "ap": ap, "fpr": fp / (fp + tn),
    })
    pd.DataFrame({
        "node": gcn_node_ids[selected_mask], "label": labels,
        "score": scores, "prediction": predictions,
    }).to_csv(GCN_OUT / f"gcn_{split_name}_predictions.csv", index=False)

    print("\nGCN HEAD —", split_name.upper())
    print(metrics.classification_report(
        labels, predictions, labels=[0, 1],
        target_names=["isp=0", "Phishing-labelled"], digits=4, zero_division=0
    ))

    plt.figure(figsize=(6, 5))
    metrics.ConfusionMatrixDisplay(gcn_cm, display_labels=["isp=0", "Phishing-labelled"]).plot(
        ax=plt.gca(), values_format="d", colorbar=False
    )
    plt.title(f"GCN head — {split_name} — threshold 0.5")
    save_gcn_figure(f"gcn_{split_name}_confusion.png")

    precision, recall, gcn_unused = metrics.precision_recall_curve(labels, scores)
    plt.figure(figsize=(6, 5))
    plt.step(recall, precision, where="post", label=f"GCN head: AP={ap:.4f}")
    plt.axhline(float(labels.mean()), linestyle="--", label="Positive proportion")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.xlim(0, 1)
    plt.ylim(0, 1.05)
    plt.title(f"GCN head — {split_name} Precision–Recall")
    plt.legend()
    plt.grid(alpha=0.25)
    save_gcn_figure(f"gcn_{split_name}_pr.png")

gcn_metrics_table = pd.DataFrame(gcn_metric_rows).set_index("split")
print(gcn_metrics_table.round(4).to_string())
gcn_metrics_table.to_csv(GCN_OUT / "gcn_train_validation_metrics.csv")

### 13.9 Export frozen embeddings for the RF2 stage

Generate embeddings from the **restored best checkpoint with dropout disabled**. Save the wallet ID with every embedding, and merge RF2 input by ID rather than assuming equal row orders.

For the current architecture, $H$ has 32 columns. RF2 input is the **unchanged raw 12 RF features** concatenated with these 32 embeddings, i.e. 44 features. GCN-only log/scaling must not silently replace RF1's raw input.

Exporting embeddings for test/context nodes does not use their labels; do not compute test performance until the complete comparison protocol is frozen. Supervised embeddings can overfit the train nodes, so final claims depend on held-out evaluation, not a high RF2 training score.

In [ ]:
# Already restored to the best validation checkpoint above.
gcn.eval()
for parameter in gcn.parameters():
    parameter.requires_grad_(False)

with torch.no_grad():
    gcn_H = gcn.encode(gcn_x, gcn_adj).cpu().numpy()
assert gcn_H.shape == (GCN_N, gcn_config["embedding_dim"])
assert np.isfinite(gcn_H).all()

gcn_embedding_cols = [f"gcn_h_{gcn_i:02d}" for gcn_i in range(gcn_H.shape[1])]
gcn_embeddings = pd.DataFrame(gcn_H, columns=gcn_embedding_cols)
gcn_embeddings.insert(0, "node", gcn_node_ids)
gcn_embeddings.to_csv(GCN_OUT / "gcn_embeddings.csv", index=False)
np.savez_compressed(
    GCN_OUT / "gcn_embeddings.npz",
    node_ids=gcn_node_ids, H=gcn_H,
    train_mask=gcn_train_np, val_mask=gcn_val_np, test_mask=gcn_test_np,
)

print("Embedding shape:", gcn_H.shape)
print("Future RF2 feature count:", len(GCN_FEATURE_COLS) + gcn_H.shape[1])
print("Saved:", GCN_OUT / "gcn_embeddings.csv")
print("Saved:", GCN_OUT / "gcn_best_state.pt")
print("RF1 was not changed. RF2 has not been trained. No test metrics were computed.")

### 13.10 Design notes and sources

The feature extractor summarized each selected wallet's transactions in the full released network. The notebook does not know the time when each phishing label became available. It therefore supports a retrospective benchmark, not an early-warning or prospective fraud-detection claim. One-hop context extraction is a computational approximation; two GCN layers do not recover omitted outside edges.

Sources:

1. EPTransNet README, distributed with the dataset: accounts are nodes, transactions are directed multiedges, `isp=1` means phishing-labelled and `isp=0` otherwise. Dataset: https://www.kaggle.com/datasets/xblock/ethereum-phishing-transaction-network
2. Kipf and Welling, *Semi-Supervised Classification with Graph Convolutional Networks*: https://arxiv.org/abs/1609.02907
3. Documented GCN node-wise normalization/source-to-target convention: https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GCNConv.html
4. PyTorch sparse multiplication: https://docs.pytorch.org/docs/stable/generated/torch.sparse.mm.html
5. PyTorch CrossEntropyLoss: https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
6. StandardScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
7. Average Precision: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html
8. PyTorch reproducibility: https://docs.pytorch.org/docs/stable/notes/randomness.html

## 14. RF2, fair RF tuning, and final comparison

This part trains RF2, tunes both forests under matched conditions and compares them. It does not retrain the GCN, change its checkpoint, change node splits, or overwrite the original `rf` / `rf_baseline` variables.

Run Sections **14–20** in order after the GCN embeddings have been exported. Only these saved inputs are required for this part:

```text
/content/node_features_raw.csv
/content/node_splits.csv
/content/gcn_outputs/gcn_embeddings.csv
```

The output directory is `/content/rf_comparison`. A new Colab runtime does not keep these files, and notebook outputs do not contain the CSVs or fitted models, so after a restart run the earlier sections again first.

**Predeclared experiment:**

| Model | Input | RF settings |
|---|---|---|
| RF1 initial | 12 unchanged raw features, $X$ | Original 300-tree configuration |
| RF2 initial | $[X \Vert H]$: raw features + 32 frozen GCN embeddings | Same initial RF configuration |
| RF1 tuned | $X$ | Best validation AP among 18 candidates |
| RF2 tuned | $[X \Vert H]$ | Independently selected from the same 18 candidates |

Both forests train on the **same 1,630 training accounts**, validate on the same 1,050 accounts, and are finally tested on the same 3,675 accounts. Context nodes are not additional supervised examples. The GCN already used validation AP to select its checkpoint; this is additional model-selection effort for the hybrid, not an equal total-compute comparison.

**Selection rules, set before opening test metrics:** use validation **Average Precision (AP)** for RF hyperparameters; keep the first candidate in a deterministic grid order on an AP tie. Report threshold **0.5** for all four models. As a separately labelled secondary analysis, tune one threshold per *tuned* forest to maximise validation phishing-class F1, choosing the higher threshold on a tie. The two selected thresholds may differ. Neither threshold is selected on test.

The final test cell runs only after model/threshold settings are saved. The pipeline is not changed based on its test results. This is a fixed-split benchmark, not proof of real-world deployment performance or statistical significance.

### 14.1 Imports and frozen experiment settings

All imports for this part are grouped here. Some repeat earlier imports so that this section can also resume from the three saved CSVs. This uses scikit-learn `ParameterGrid` with the **existing validation set**, not automatic cross-validation over frozen supervised GCN embeddings. A valid end-to-end cross-validation experiment would have to refit the GCN inside each fold.

The 18 RF configurations are proposed controlled experiments, not guaranteed optimal settings. Raw RF features remain unscaled. Using the same `max_features` rule does not imply the same absolute number of split candidates when the input has 12 versus 44 columns.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import shutil
import time
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid

RC_DATA_DIR = Path('/content')
RC_OUT = RC_DATA_DIR / 'rf_comparison'
for folder in ['models', 'tables', 'predictions', 'figures']:
    (RC_OUT / folder).mkdir(parents=True, exist_ok=True)

RC_SEED = 42
RC_N_JOBS = -1  # Candidates are fitted sequentially; parallelism is inside each forest.
RC_RUN_FINAL_TEST = True  # Set False BEFORE running if you are still changing the pipeline.

RC_RAW_COLS = [
    'num_in', 'num_out', 'unique_senders', 'unique_receivers',
    'total_in', 'total_out', 'mean_in', 'mean_out',
    'max_in', 'max_out', 'activity_duration', 'net_amount',
]
RC_FIXED_RF = dict(criterion='gini', bootstrap=True, class_weight=None,
                   random_state=RC_SEED, n_jobs=RC_N_JOBS)
RC_INITIAL = dict(n_estimators=300, max_depth=None,
                  min_samples_leaf=1, max_features='sqrt')
RC_GRID = {
    'n_estimators': [300],
    'max_depth': [None, 8, 16],
    'min_samples_leaf': [1, 3, 8],
    'max_features': ['sqrt', 0.5],
}
RC_CANDIDATES = list(ParameterGrid(RC_GRID))
RC_EXPECTED_COUNTS = {
    ('train', 0): 815, ('train', 1): 815,
    ('val', 0): 875, ('val', 1): 175,
    ('test', 0): 3500, ('test', 1): 175,
}
assert len(RC_CANDIDATES) == 18
assert RC_INITIAL in RC_CANDIDATES

print('RF configurations per model:', len(RC_CANDIDATES))
print('Selection: validation AP; no test-based tuning.')
print('Test will run after protocol freeze:', RC_RUN_FINAL_TEST)
print('Outputs:', RC_OUT)
print('scikit-learn:', sklearn.__version__)

### 14.2 Match raw features, labels and embeddings by wallet ID

Do not concatenate unrelated row orders. Require a one-to-one join by `node`, verify the original split, and preserve the sorted wallet order for both RF inputs. Only the named feature columns and `gcn_h_00`…`gcn_h_31` enter a forest; node ID, `isp`, label, seed membership, and split never enter the feature matrix.

GCN embeddings are read from the checkpoint selected earlier with validation AP and dropout disabled. No GCN updates occur in this part. Supervised training embeddings can overfit training nodes; high RF2 training performance alone is not evidence of generalisation.

In [ ]:
def rc_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

rc_files = {
    'features': RC_DATA_DIR / 'node_features_raw.csv',
    'splits': RC_DATA_DIR / 'node_splits.csv',
    'embeddings': RC_DATA_DIR / 'gcn_outputs' / 'gcn_embeddings.csv',
}
for name, path in rc_files.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Restore the saved {name} file first.')

rc_raw = pd.read_csv(rc_files['features'], dtype={'node': str})
rc_splits = pd.read_csv(rc_files['splits'], dtype={'node': str})
rc_h = pd.read_csv(rc_files['embeddings'], dtype={'node': str})

for name, frame in [('features', rc_raw), ('splits', rc_splits), ('embeddings', rc_h)]:
    if 'node' not in frame or frame['node'].isna().any() or frame['node'].duplicated().any():
        raise ValueError(f'Missing or duplicate node IDs in {name}.')
    if frame['node'].str.strip().ne(frame['node']).any():
        raise ValueError(f'Whitespace in wallet IDs in {name}; inspect instead of guessing.')

if not set(RC_RAW_COLS).issubset(rc_raw.columns):
    raise ValueError('The raw table is missing expected feature columns.')
if not {'label', 'split'}.issubset(rc_splits.columns):
    raise ValueError('node_splits.csv needs node, label and split.')
if not rc_splits['label'].isin([0, 1]).all():
    raise ValueError('Labels must be 0/1; never fill missing labels with 0.')
if not rc_splits['split'].isin(['train', 'val', 'test']).all():
    raise ValueError('Unexpected split value.')
if rc_splits.groupby(['split', 'label']).size().to_dict() != RC_EXPECTED_COUNTS:
    raise ValueError('Split counts changed. Do not silently resample the benchmark.')
if not rc_splits['node'].isin(rc_raw['node']).all():
    raise ValueError('Some supervised accounts have no raw features.')
if set(rc_raw['node']) != set(rc_h['node']):
    raise ValueError('Embedding IDs and feature IDs differ. Check the GCN export.')

RC_H_COLS = [f'gcn_h_{i:02d}' for i in range(32)]
if set(rc_h.columns) != {'node', *RC_H_COLS}:
    raise ValueError('Expected node and exactly 32 exported GCN embedding columns.')

# Check saved metadata before replacing it with the authoritative split CSV.
for col in ['label', 'split']:
    if col in rc_raw.columns:
        old = rc_raw.set_index('node')[col].reindex(rc_splits['node'])
        if old.isna().any() or not np.all(old.to_numpy() == rc_splits[col].to_numpy()):
            raise ValueError(f'Conflicting {col} metadata in the feature file.')

rc_table = (
    rc_raw[['node'] + RC_RAW_COLS]
    .merge(rc_h[['node'] + RC_H_COLS], on='node', validate='one_to_one')
    .merge(rc_splits[['node', 'label', 'split']], on='node', how='left', validate='one_to_one')
    .sort_values('node', kind='stable').reset_index(drop=True)
)
rc_table['split'] = rc_table['split'].fillna('context')
RC_HYBRID_COLS = RC_RAW_COLS + RC_H_COLS
rc_table[RC_HYBRID_COLS] = rc_table[RC_HYBRID_COLS].apply(pd.to_numeric, errors='raise')
if not np.isfinite(rc_table[RC_HYBRID_COLS].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite input; fix the data audit before fitting.')
if rc_table.loc[rc_table['split'].eq('context'), 'label'].notna().any():
    raise ValueError('Context nodes must not acquire supervised labels here.')

rc_frames = {
    split: rc_table.loc[rc_table['split'].eq(split)].reset_index(drop=True)
    for split in ['train', 'val', 'test']
}
rc_inputs = {
    'RF1': {split: frame[RC_RAW_COLS].copy() for split, frame in rc_frames.items()},
    'RF2': {split: frame[RC_HYBRID_COLS].copy() for split, frame in rc_frames.items()},
}
rc_y_train = rc_frames['train']['label'].to_numpy(dtype=int)
rc_y_val = rc_frames['val']['label'].to_numpy(dtype=int)
# Do not construct/use test targets for any training or selection function below.

rc_input_hashes = {name: rc_sha256(path) for name, path in rc_files.items()}
(RC_OUT / 'input_manifest.json').write_text(
    json.dumps({'sha256': rc_input_hashes, 'paths': {k: str(v) for k, v in rc_files.items()},
                'raw_columns': RC_RAW_COLS, 'embedding_columns': RC_H_COLS}, indent=2),
    encoding='utf-8'
)
for split in ['train', 'val', 'test']:
    print(split, '| RF1:', rc_inputs['RF1'][split].shape,
          '| RF2:', rc_inputs['RF2'][split].shape)
print('Joins verified; original rf and GCN parameters are unchanged.')

### 14.3 Shared evaluation and plotting helpers

`rc_metrics` compares predictions against labels without fitting a model. AP and ROC-AUC use continuous scores; precision, recall, F1 and confusion counts use a stated threshold. **AP is not detection accuracy or interpolated PR-AUC.** All figures are separately saved and titled with the split, model and threshold where applicable.

In [ ]:
def rc_scores(model, X):
    positive_column = list(model.classes_).index(1)
    return model.predict_proba(X)[:, positive_column]


def rc_model_fingerprint(model):
    # Hash the learned numeric tree structures rather than pickle bytes.
    # Loading and re-saving an equivalent model can change serialization bytes.
    digest = hashlib.sha256(json.dumps(model.get_params(), sort_keys=True).encode())
    digest.update(np.asarray(model.classes_, dtype=np.int64).tobytes())
    digest.update(json.dumps(list(model.feature_names_in_)).encode())
    for estimator in model.estimators_:
        tree = estimator.tree_
        for attr in ['children_left', 'children_right', 'feature', 'threshold',
                     'value', 'missing_go_to_left']:
            if hasattr(tree, attr):
                array = np.ascontiguousarray(getattr(tree, attr))
                digest.update(attr.encode())
                digest.update(str((array.dtype.str, array.shape)).encode())
                digest.update(array.tobytes())
    return digest.hexdigest()


def rc_metrics(y, scores, threshold=0.5):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.shape != scores.shape or not np.isfinite(scores).all():
        raise ValueError('Invalid predictions or label alignment.')
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = metrics.confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'samples': len(y), 'positive_fraction': float(y.mean()),
        'threshold': float(threshold),
        'AP': float(metrics.average_precision_score(y, scores)),
        'ROC_AUC': float(metrics.roc_auc_score(y, scores)),
        'precision': float(metrics.precision_score(y, pred, zero_division=0)),
        'recall': float(metrics.recall_score(y, pred, zero_division=0)),
        'F1_phishing': float(metrics.f1_score(y, pred, zero_division=0)),
        'macro_F1': float(metrics.f1_score(y, pred, average='macro', zero_division=0)),
        'FPR': float(fp / (fp + tn)) if fp + tn else np.nan,
        'accuracy': float(metrics.accuracy_score(y, pred)),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }


def rc_save_figure(name):
    plt.tight_layout()
    plt.savefig(RC_OUT / 'figures' / f'{name}.png', dpi=250, bbox_inches='tight')
    plt.show()
    plt.close()


def rc_plot_pr(y, score_map, split, stage):
    plt.figure(figsize=(7, 5))
    styles = ['-', '--', '-.', ':']
    for index, (name, scores) in enumerate(score_map.items()):
        p, r, _ = metrics.precision_recall_curve(y, scores)
        ap = metrics.average_precision_score(y, scores)
        plt.step(r, p, where='post', linestyle=styles[index % len(styles)],
                 label=f'{name}: AP={ap:.4f}')
    plt.axhline(float(np.mean(y)), linestyle=':',
                label=f'Positive fraction={np.mean(y):.3f}')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'{split} — Precision–Recall ({stage})')
    plt.xlim(0, 1); plt.ylim(0, 1.03)
    plt.grid(alpha=0.25); plt.legend(fontsize=9)
    rc_save_figure(f'{split.lower()}_{stage}_pr')


def rc_plot_cm(y, scores, name, split, threshold, policy):
    cm = metrics.confusion_matrix(y, scores >= threshold, labels=[0, 1])
    plt.figure(figsize=(6, 5))
    metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=['isp=0', 'Phishing-labelled']
    ).plot(ax=plt.gca(), colorbar=False, values_format='d')
    plt.title(f'{name} — {split}\n{policy}; threshold={threshold:.4f}')
    rc_save_figure(f'{split.lower()}_{name}_{policy}_cm')


def rc_plot_metric_bars(table, split, metric):
    rows = table.loc[(table['split'] == split) & (table['threshold_policy'] == 'fixed_0.5')]
    plt.figure(figsize=(8, 4.8))
    bars = plt.bar(rows['model'], rows[metric])
    for bar, value in zip(bars, rows[metric]):
        plt.text(bar.get_x() + bar.get_width()/2, value + 0.01, f'{value:.4f}',
                 ha='center', va='bottom', fontsize=9)
    plt.ylim(0, 1.10); plt.ylabel(metric)
    plt.xticks(rotation=15)
    suffix = '' if metric in ['AP', 'ROC_AUC'] else ' (threshold 0.5)'
    plt.title(f'{split} — {metric}: before/after RF tuning{suffix}')
    rc_save_figure(f'{split.lower()}_{metric}_before_after')

### 14.4 Train RF2 with the original RF1 configuration

We create `rf1_initial` and `rf2_initial` as separate objects. RF1 is **refitted reproducibly on the same saved train rows** to make the comparison resumable from CSVs, without changing the original `rf`. The original results above are retained, not replaced. A schema/order/configuration mismatch should be investigated if the new initial RF1 does not reproduce the earlier result under the same environment.

RF2 is trained from scratch on 44 columns; it does **not** extend or fine-tune RF1's existing trees. The GCN remains frozen and is not called by either RF fit.

In [ ]:
rc_initial_models = {}
rc_initial_times = {}
rc_initial_val_scores = {}
rc_initial_rows = []

for branch in ['RF1', 'RF2']:
    start = time.perf_counter()
    model = RandomForestClassifier(**RC_FIXED_RF, **RC_INITIAL)
    model.fit(rc_inputs[branch]['train'], rc_y_train)
    rc_initial_times[branch] = time.perf_counter() - start
    rc_initial_models[branch] = model

    for split, y in [('train', rc_y_train), ('val', rc_y_val)]:
        scores = rc_scores(model, rc_inputs[branch][split])
        rc_initial_rows.append({'model': branch + '_initial', 'split': split,
                                'threshold_policy': 'fixed_0.5', **rc_metrics(y, scores)})
        if split == 'val':
            rc_initial_val_scores[branch] = scores
    print(branch, 'initial fit complete:', round(rc_initial_times[branch], 2), 'seconds')

rf1_initial = rc_initial_models['RF1']
rf2_initial = rc_initial_models['RF2']
rc_initial_results = pd.DataFrame(rc_initial_rows)
rc_initial_results.to_csv(RC_OUT / 'tables' / 'initial_train_validation.csv', index=False)
display(rc_initial_results.round(4))
rc_plot_pr(rc_y_val, rc_initial_val_scores, 'Validation', 'initial')
print('No test scores have been used.')

## 15. Tune RF1 and RF2 separately with the same 18 RF candidates

Each forest is fitted on **train only**, then scored on **validation only**. A saved initial model is reused when its configuration appears in the grid. Candidates run sequentially, so the notebook does not keep 36 fitted candidate forests in memory. Trial metrics are saved after every candidate.

Selection is by **validation AP**, not train F1, test AP or the RF's default accuracy score. The 300-tree budget is fixed in this search; this is not an exhaustive search over all hyperparameters. The selected settings can differ between RF1 and RF2. Validation performance is a selection result, not an unbiased final estimate.

In [ ]:
def rc_tune(branch, initial_model):
    best_model, best_params, best_ap, best_trial = None, None, -np.inf, None
    trials = []
    for trial_id, params in enumerate(RC_CANDIDATES, 1):
        start = time.perf_counter()
        reused = params == RC_INITIAL
        if reused:
            candidate = initial_model
        else:
            candidate = RandomForestClassifier(**RC_FIXED_RF, **params)
            candidate.fit(rc_inputs[branch]['train'], rc_y_train)

        train_s = rc_scores(candidate, rc_inputs[branch]['train'])
        val_s = rc_scores(candidate, rc_inputs[branch]['val'])
        train_m = rc_metrics(rc_y_train, train_s)
        val_m = rc_metrics(rc_y_val, val_s)
        trials.append({
            'branch': branch, 'trial': trial_id, 'params_json': json.dumps(params, sort_keys=True),
            'train_AP': train_m['AP'], 'val_AP': val_m['AP'],
            'train_F1_at_0.5': train_m['F1_phishing'], 'val_F1_at_0.5': val_m['F1_phishing'],
            'val_precision_at_0.5': val_m['precision'], 'val_recall_at_0.5': val_m['recall'],
            'seconds': time.perf_counter() - start, 'reused_initial_model': reused,
        })
        # On equal AP, keep the first candidate in the fixed ParameterGrid order.
        if best_model is None or val_m['AP'] > best_ap + 1e-12:
            best_model, best_params = candidate, params.copy()
            best_ap, best_trial = val_m['AP'], trial_id
        pd.DataFrame(trials).to_csv(RC_OUT / 'tables' / f'{branch}_tuning.csv', index=False)
        print(f'{branch} {trial_id:02d}/{len(RC_CANDIDATES)} | '
              f'val AP={val_m["AP"]:.4f} | best={best_ap:.4f} | {params}')

    if best_model is None:
        raise RuntimeError('No candidate fitted successfully.')
    if best_ap + 1e-12 < metrics.average_precision_score(rc_y_val, rc_initial_val_scores[branch]):
        raise AssertionError('Initial candidate was not represented correctly in the search.')
    return best_model, best_params, pd.DataFrame(trials), best_trial


rf1_tuned, rf1_best_params, rc_rf1_trials, rc_rf1_best_trial = rc_tune('RF1', rf1_initial)
rf2_tuned, rf2_best_params, rc_rf2_trials, rc_rf2_best_trial = rc_tune('RF2', rf2_initial)

rc_best_params = {'RF1_tuned': rf1_best_params, 'RF2_tuned': rf2_best_params}
rc_best_param_table = pd.DataFrame([
    {'model': 'RF1_tuned', **rf1_best_params, 'best_trial': rc_rf1_best_trial,
     'validation_AP': rc_rf1_trials['val_AP'].max()},
    {'model': 'RF2_tuned', **rf2_best_params, 'best_trial': rc_rf2_best_trial,
     'validation_AP': rc_rf2_trials['val_AP'].max()},
])
display(rc_best_param_table)
rc_best_param_table.to_csv(RC_OUT / 'tables' / 'selected_RF_parameters.csv', index=False)

plt.figure(figsize=(8, 4.8))
plt.plot(rc_rf1_trials['trial'], rc_rf1_trials['val_AP'], marker='o', label='RF1')
plt.plot(rc_rf2_trials['trial'], rc_rf2_trials['val_AP'], marker='s', label='RF2')
plt.xticks(range(1, len(RC_CANDIDATES) + 1))
plt.xlabel('RF configuration ID (not epoch)'); plt.ylabel('Validation AP')
plt.title('Same RF search space — separate validation-based selection')
plt.grid(alpha=0.25); plt.legend()
rc_save_figure('rf_hyperparameter_search')

## 16. Select decision thresholds using validation only

The **primary ranking comparison is AP**, independent of the classification threshold. First report both models at the common threshold 0.5. A **secondary** comparison applies the same procedure to both tuned forests: select a threshold that maximises phishing-class F1 on validation.

The validation class ratio is 1:5, whereas test is 1:20. A validation-F1-optimal threshold is **not guaranteed** to be optimal under the test or real-world prevalence. This notebook does not retune thresholds on test, use test weights to optimise them, or claim calibrated probabilities. Threshold tuning changes predictions, not the learned trees or AP.

In [ ]:
def rc_select_threshold(y, scores):
    precision, recall, thresholds = metrics.precision_recall_curve(y, scores)
    denom = precision[:-1] + recall[:-1]
    f1 = np.divide(2 * precision[:-1] * recall[:-1], denom,
                   out=np.zeros_like(denom), where=denom > 0)
    table = pd.DataFrame({'threshold': thresholds, 'precision': precision[:-1],
                          'recall': recall[:-1], 'F1_phishing': f1})
    if table.empty:
        raise ValueError('No validation thresholds available.')
    tied = np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))
    chosen = int(tied[-1])  # Increasing thresholds; prefer fewer alerts on a tie.
    return float(thresholds[chosen]), table


rc_models = {
    'RF1_initial': rf1_initial, 'RF2_initial': rf2_initial,
    'RF1_tuned': rf1_tuned, 'RF2_tuned': rf2_tuned,
}
rc_branch = {name: name.split('_')[0] for name in rc_models}
rc_val_scores = {
    name: rc_scores(model, rc_inputs[rc_branch[name]]['val'])
    for name, model in rc_models.items()
}
rc_thresholds = {}
rc_threshold_tables = {}
for name in ['RF1_tuned', 'RF2_tuned']:
    rc_thresholds[name], sweep = rc_select_threshold(rc_y_val, rc_val_scores[name])
    rc_threshold_tables[name] = sweep
    sweep.to_csv(RC_OUT / 'tables' / f'{name}_validation_thresholds.csv', index=False)
    print(name, '| selected threshold:', round(rc_thresholds[name], 6))

plt.figure(figsize=(8, 4.8))
for name, sweep in rc_threshold_tables.items():
    plt.plot(sweep['threshold'], sweep['F1_phishing'], label=name)
    chosen = rc_thresholds[name]
    plt.scatter([chosen], [rc_metrics(rc_y_val, rc_val_scores[name], chosen)['F1_phishing']],
                label=f'{name} selected: {chosen:.3f}')
plt.axvline(0.5, linestyle='--', label='Original threshold 0.5')
plt.xlabel('Decision threshold'); plt.ylabel('Validation phishing F1')
plt.title('Threshold selection on validation only')
plt.legend(fontsize=9); plt.grid(alpha=0.25)
rc_save_figure('validation_threshold_selection')

### 16.1 Train and validation comparison before opening the test results

The fixed-threshold table contains the four predeclared configurations; the second table shows the two tuned forests with validation-selected thresholds. All predictions retain wallet IDs for later paired error analysis. Training scores are diagnostic only. No model is refitted on train+validation after threshold selection, so the selected threshold still refers to the same fitted model.

In [ ]:
rc_pretest_rows = []
rc_prediction_frames = {}
for split, y in [('train', rc_y_train), ('val', rc_y_val)]:
    predictions = pd.DataFrame({'node': rc_frames[split]['node'], 'label': y})
    for name, model in rc_models.items():
        scores = rc_scores(model, rc_inputs[rc_branch[name]][split])
        predictions[name + '_score'] = scores
        predictions[name + '_pred_0.5'] = (scores >= 0.5).astype(int)
        rc_pretest_rows.append({'model': name, 'split': split,
                               'threshold_policy': 'fixed_0.5', **rc_metrics(y, scores)})
        if name in rc_thresholds:
            t = rc_thresholds[name]
            predictions[name + '_pred_val_f1'] = (scores >= t).astype(int)
            rc_pretest_rows.append({'model': name, 'split': split,
                                   'threshold_policy': 'validation_F1', **rc_metrics(y, scores, t)})
    predictions.to_csv(RC_OUT / 'predictions' / f'{split}_predictions.csv', index=False)
    rc_prediction_frames[split] = predictions

rc_pretest_results = pd.DataFrame(rc_pretest_rows)
rc_pretest_results.to_csv(RC_OUT / 'tables' / 'train_validation_comparison.csv', index=False)
print('COMMON THRESHOLD = 0.5')
display(rc_pretest_results.query("threshold_policy == 'fixed_0.5'").round(4))
print('SECONDARY ANALYSIS: VALIDATION-SELECTED F1 THRESHOLDS')
display(rc_pretest_results.query("threshold_policy == 'validation_F1'").round(4))
rc_plot_pr(rc_y_val, rc_val_scores, 'Validation', 'all_models')
rc_plot_metric_bars(rc_pretest_results, 'val', 'AP')
rc_plot_metric_bars(rc_pretest_results, 'val', 'F1_phishing')

for split, y in [('train', rc_y_train), ('val', rc_y_val)]:
    for name in ['RF1_tuned', 'RF2_tuned']:
        rc_plot_cm(y, rc_prediction_frames[split][name + '_score'].to_numpy(),
                   name, split, 0.5, 'fixed_0.5')

## 17. Freeze and save the selected models before test evaluation

This saves all four forests, exact input feature orders, the search space, chosen RF parameters, chosen thresholds, software versions, and hashes of the input files. The GCN weights/embeddings and existing split remain fixed. Only use `joblib.load` on trusted files.

The input/model hashes bind the final test report to this selection. If a previous test marker in this output directory belongs to a different selection, the notebook stops rather than silently treating a changed experiment as an untouched test. The marker is an audit aid, not a substitute for good evaluation practice.

In [ ]:
rc_model_paths = {}
rc_model_hashes = {}
rc_staged_models = {}
rc_model_fingerprints = {}
for name, model in rc_models.items():
    path = RC_OUT / 'models' / f'{name}.joblib'
    staging = RC_OUT / 'models' / f'{name}.pending.joblib'
    joblib.dump(model, staging)
    rc_staged_models[name] = staging
    rc_model_paths[name] = str(path)
    rc_model_hashes[name] = rc_sha256(staging)
    rc_model_fingerprints[name] = rc_model_fingerprint(model)

rc_gcn_info_path = RC_DATA_DIR / 'gcn_outputs' / 'gcn_config.json'
rc_gcn_info = json.loads(rc_gcn_info_path.read_text()) if rc_gcn_info_path.is_file() else None
rc_protocol = {
    'input_sha256': rc_input_hashes,
    'learned_model_sha256': rc_model_fingerprints,
    'raw_features': RC_RAW_COLS,
    'embedding_features': RC_H_COLS,
    'rf1_n_features': len(RC_RAW_COLS),
    'rf2_n_features': len(RC_HYBRID_COLS),
    'initial_settings': RC_INITIAL,
    'fixed_rf_settings': RC_FIXED_RF,
    'grid': RC_GRID,
    'candidates_per_forest': len(RC_CANDIDATES),
    'selection_metric': 'validation_AP',
    'AP_tie_break': 'first candidate in deterministic ParameterGrid order',
    'selected_parameters': rc_best_params,
    'selected_validation_F1_thresholds': rc_thresholds,
    'threshold_tie_break': 'higher threshold on equal validation phishing F1',
    'primary_comparison': 'RF2_tuned versus RF1_tuned AP on the same test rows',
    'threshold_comparisons': ['fixed_0.5', 'validation_F1'],
    'refit_on_train_plus_validation': False,
    'GCN_retrained_in_this_section': False,
    'GCN_selection': rc_gcn_info,
    'software': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'sklearn': sklearn.__version__},
}
rc_protocol_json = json.dumps(rc_protocol, sort_keys=True, indent=2, allow_nan=False)
rc_protocol_hash = hashlib.sha256(rc_protocol_json.encode()).hexdigest()
rc_marker_path = RC_OUT / 'test_evaluated.json'
if rc_marker_path.is_file():
    old = json.loads(rc_marker_path.read_text())
    if old.get('protocol_sha256') != rc_protocol_hash:
        for staging in rc_staged_models.values():
            staging.unlink(missing_ok=True)
        raise RuntimeError(
            'Test results already exist for a DIFFERENT protocol in this directory. '
            'Do not retune using test feedback or present a reused test as unseen. '
            'Keep the old report and document any further analysis as exploratory.'
        )
for name, staging in rc_staged_models.items():
    staging.replace(rc_model_paths[name])
(RC_OUT / 'selection_protocol.json').write_text(rc_protocol_json, encoding='utf-8')
(RC_OUT / 'model_files.json').write_text(
    json.dumps({'paths': rc_model_paths, 'file_sha256': rc_model_hashes}, indent=2), encoding='utf-8'
)
rc_protocol_frozen = True
print('Selection frozen:', rc_protocol_hash[:16])
print('Saved models and feature orders. GCN is unchanged.')
print('Continue to Section 18 for the predeclared final test evaluation.')

## 18. Final held-out test comparison — no more tuning

This is the **first test-metric computation in the pipeline**. The four named models and both threshold policies were fixed before this cell. Testing the initial configurations is a predeclared secondary comparison, not a search for a new winner.

`RC_RUN_FINAL_TEST=True` runs this section after the protocol is frozen. Set it to `False` in Section 14.1 before execution if the experiment is still under development. Identical reruns reload the saved test predictions; a different protocol is rejected when a prior test marker exists. Do not train on test, select a new threshold from test, or choose a new GCN checkpoint from these results.

In [ ]:
rc_test_results = None
rc_test_predictions = None
if not RC_RUN_FINAL_TEST:
    print('Test is still held out. RF models and validation results are saved.')
else:
    if not rc_protocol_frozen:
        raise RuntimeError('Freeze the protocol in Section 17 first.')
    current_hashes = {k: rc_sha256(v) for k, v in rc_files.items()}
    if current_hashes != rc_input_hashes:
        raise RuntimeError('Input files changed after selection.')
    for name, path in rc_model_paths.items():
        if rc_sha256(path) != rc_model_hashes[name]:
            raise RuntimeError('A saved model changed after selection.')
        # Evaluate the exact frozen artifact, not a possibly modified in-memory variable.
        rc_models[name] = joblib.load(path)

    if rc_marker_path.is_file():
        marker = json.loads(rc_marker_path.read_text())
        if marker.get('protocol_sha256') != rc_protocol_hash:
            raise RuntimeError('Existing test results belong to another protocol.')
        rc_test_predictions = pd.read_csv(RC_OUT / 'predictions' / 'test_predictions.csv',
                                          dtype={'node': str})
        if rc_sha256(RC_OUT / 'predictions' / 'test_predictions.csv') != marker['prediction_sha256']:
            raise RuntimeError('Saved test predictions changed; inspect the audit trail.')
        print('Reusing saved predictions for the identical frozen protocol.')
    else:
        # Test targets enter metrics only here.
        rc_test_predictions = pd.DataFrame({
            'node': rc_frames['test']['node'],
            'label': rc_frames['test']['label'].to_numpy(dtype=int),
        })
        for name, model in rc_models.items():
            scores = rc_scores(model, rc_inputs[rc_branch[name]]['test'])
            rc_test_predictions[name + '_score'] = scores
            rc_test_predictions[name + '_pred_0.5'] = (scores >= 0.5).astype(int)
            if name in rc_thresholds:
                rc_test_predictions[name + '_pred_val_f1'] = (
                    scores >= rc_thresholds[name]
                ).astype(int)
        prediction_path = RC_OUT / 'predictions' / 'test_predictions.csv'
        rc_test_predictions.to_csv(prediction_path, index=False)
        rc_marker_path.write_text(json.dumps({
            'protocol_sha256': rc_protocol_hash,
            'prediction_sha256': rc_sha256(prediction_path),
            'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
            'warning': 'The test set has now been opened. No further test-guided selection.',
        }, indent=2), encoding='utf-8')

    if not np.array_equal(rc_test_predictions['node'].to_numpy(),
                          rc_frames['test']['node'].to_numpy()):
        raise ValueError('Saved test predictions no longer match the selected test accounts.')
    rc_y_test = rc_test_predictions['label'].to_numpy(dtype=int)
    if not np.array_equal(rc_y_test, rc_frames['test']['label'].to_numpy(dtype=int)):
        raise ValueError('Saved test labels conflict with the split file.')

    rc_test_rows = []
    for name in rc_models:
        scores = rc_test_predictions[name + '_score'].to_numpy()
        rc_test_rows.append({'model': name, 'split': 'test', 'threshold_policy': 'fixed_0.5',
                             **rc_metrics(rc_y_test, scores)})
        if name in rc_thresholds:
            rc_test_rows.append({'model': name, 'split': 'test', 'threshold_policy': 'validation_F1',
                                 **rc_metrics(rc_y_test, scores, rc_thresholds[name])})
    rc_test_results = pd.DataFrame(rc_test_rows)
    rc_test_results.to_csv(RC_OUT / 'tables' / 'test_comparison.csv', index=False)
    rc_all_results = pd.concat([rc_pretest_results, rc_test_results], ignore_index=True)
    rc_all_results.to_csv(RC_OUT / 'tables' / 'all_metrics.csv', index=False)

    print('FINAL TEST — COMMON THRESHOLD = 0.5')
    display(rc_test_results.query("threshold_policy == 'fixed_0.5'").round(4))
    print('FINAL TEST — THRESHOLDS SELECTED ON VALIDATION, NOT TEST')
    display(rc_test_results.query("threshold_policy == 'validation_F1'").round(4))

## 19. Final figures and paired error analysis

Compare RF1 and RF2 **on the same test accounts**, not RF1 on validation versus RF2 on test. The paired error table counts which model alone was correct at the common threshold 0.5. It is descriptive; it is not a statistical-significance test. The account graph and sampling scheme violate an independent-samples interpretation of naive uncertainty estimates.

Each chart has its own figure. Test threshold curves are deliberately not plotted for threshold selection. PR/ROC curves describe ranking, not a recommended operating threshold.

In [ ]:
if rc_test_results is not None:
    rc_test_score_map = {
        name: rc_test_predictions[name + '_score'].to_numpy() for name in rc_models
    }
    rc_plot_pr(rc_y_test, rc_test_score_map, 'Test', 'all_models')
    rc_plot_metric_bars(rc_test_results, 'test', 'AP')
    rc_plot_metric_bars(rc_test_results, 'test', 'F1_phishing')

    plt.figure(figsize=(7, 5))
    for name in ['RF1_tuned', 'RF2_tuned']:
        scores = rc_test_score_map[name]
        fpr, tpr, _ = metrics.roc_curve(rc_y_test, scores)
        auc = metrics.roc_auc_score(rc_y_test, scores)
        plt.plot(fpr, tpr, label=f'{name}: ROC-AUC={auc:.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', label='No-skill reference')
    plt.xlabel('False positive rate'); plt.ylabel('Recall / true positive rate')
    plt.title('Test — tuned forests ROC comparison')
    plt.xlim(0, 1); plt.ylim(0, 1.03); plt.legend(); plt.grid(alpha=0.25)
    rc_save_figure('test_tuned_roc')

    for name in ['RF1_tuned', 'RF2_tuned']:
        rc_plot_cm(rc_y_test, rc_test_score_map[name], name, 'Test', 0.5, 'fixed_0.5')
        rc_plot_cm(rc_y_test, rc_test_score_map[name], name, 'Test',
                   rc_thresholds[name], 'validation_F1')

    first = rc_test_predictions['RF1_tuned_pred_0.5'].to_numpy() == rc_y_test
    second = rc_test_predictions['RF2_tuned_pred_0.5'].to_numpy() == rc_y_test
    paired_counts = pd.DataFrame({
        'outcome': ['Both correct', 'Only RF1 correct', 'Only RF2 correct', 'Both wrong'],
        'accounts': [int((first & second).sum()), int((first & ~second).sum()),
                     int((~first & second).sum()), int((~first & ~second).sum())],
    })
    paired_counts.to_csv(RC_OUT / 'tables' / 'test_paired_errors.csv', index=False)
    print('PAIRED TEST OUTCOMES — THRESHOLD 0.5')
    display(paired_counts)
    error_table = rc_test_predictions.copy()
    error_table['RF1_correct_at_0.5'] = first
    error_table['RF2_correct_at_0.5'] = second
    error_table.loc[~(first & second)].to_csv(
        RC_OUT / 'tables' / 'test_error_accounts.csv', index=False
    )
else:
    print('Test plots skipped because the final test has not been opened.')

## 20. Post-hoc analysis of the GCN embeddings (exploratory)

The analyses below look inside the frozen RF2 pipeline and run one control experiment. They use **train and validation data only**. The test set is not touched, because the final test protocol was frozen in Section 17; treat everything here as exploratory.

| Question | Section |
|---|---|
| Which raw features does each of the 32 embedding columns follow? | 20.1 |
| Does RF2 actually use the embedding columns? | 20.2 |
| Do the edges matter? Compare the GCN operator `P` with the identity matrix `I` | 20.3 |
| How much does machine learning add over one simple rule on a single raw feature? | 20.4 |

Run Sections 1–19 first: these cells reuse their variables (`rc_table`, `rc_frames`, `rc_inputs`, `rf1_tuned`, `rf2_tuned`, `gcn_x`, `gcn_adj`, ...).

### 20.1 Correlation between embedding columns and the 12 raw features

Spearman rank correlation over all 44,125 wallets (no labels are used), because the raw features are heavy-tailed. A column that is zero for almost every wallet is a "dead" ReLU unit. The last column of the table is a standardised mean difference on **training** wallets only: a positive value means the column is higher for phishing-labelled wallets.

In [ ]:
corr_all = rc_table[RC_RAW_COLS + RC_H_COLS].corr(method='spearman')
corr_block = corr_all.loc[RC_H_COLS, RC_RAW_COLS]

train_frame = rc_frames['train']
is_phish = train_frame['label'].to_numpy() == 1

corr_rows = []
for col in RC_H_COLS:
    zero_fraction = float((rc_table[col] == 0).mean())
    rho = corr_block.loc[col].dropna()
    ranked = rho.reindex(rho.abs().sort_values(ascending=False).index)
    values = train_frame[col].to_numpy(dtype=float)
    spread = values.std()
    lean = (values[is_phish].mean() - values[~is_phish].mean()) / spread if spread > 0 else np.nan
    corr_rows.append({
        'column': col,
        'zero_fraction': round(zero_fraction, 3),
        'strongest_feature': ranked.index[0] if len(ranked) else 'constant column',
        'rho_1': round(float(ranked.iloc[0]), 3) if len(ranked) else np.nan,
        'second_feature': ranked.index[1] if len(ranked) > 1 else '',
        'rho_2': round(float(ranked.iloc[1]), 3) if len(ranked) > 1 else np.nan,
        'phishing_lean_train': round(float(lean), 3) if np.isfinite(lean) else np.nan,
    })
corr_table = pd.DataFrame(corr_rows)
display(corr_table)
corr_table.to_csv(RC_OUT / 'tables' / 'embedding_feature_correlation.csv', index=False)

plt.figure(figsize=(9, 9))
plt.imshow(corr_block.to_numpy(dtype=float), cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='Spearman correlation')
plt.xticks(range(len(RC_RAW_COLS)), RC_RAW_COLS, rotation=60, ha='right')
plt.yticks(range(len(RC_H_COLS)), RC_H_COLS)
plt.title('GCN embedding columns versus the 12 raw features')
rc_save_figure('embedding_feature_correlation')

dead_columns = corr_table.loc[corr_table['zero_fraction'] > 0.99, 'column'].tolist()
print('Columns that are zero for more than 99% of wallets:', dead_columns if dead_columns else 'none')

### 20.2 Feature importance of the tuned RF2 (44 columns)

Two measures: the **impurity importance** stored in the fitted forest (fast, but biased towards columns with many distinct values), and the **permutation importance** on the validation wallets, i.e. the drop in Average Precision when one column is shuffled (correlated columns can share the credit). The test set is not used.

In [ ]:
from sklearn.inspection import permutation_importance

rf2_names = list(rf2_tuned.feature_names_in_)
impurity = pd.Series(rf2_tuned.feature_importances_, index=rf2_names)
perm = permutation_importance(
    rf2_tuned, rc_inputs['RF2']['val'], rc_y_val,
    scoring='average_precision', n_repeats=10, random_state=RC_SEED, n_jobs=1,
)
perm_mean = pd.Series(perm.importances_mean, index=rf2_names)
perm_std = pd.Series(perm.importances_std, index=rf2_names)

group_table = pd.DataFrame({
    'group': ['12 raw features', '32 GCN embedding columns'],
    'total_impurity_importance': [impurity[RC_RAW_COLS].sum(), impurity[RC_H_COLS].sum()],
    'mean_per_column': [impurity[RC_RAW_COLS].mean(), impurity[RC_H_COLS].mean()],
}).round(4)
display(group_table)

importance_table = pd.DataFrame({
    'impurity': impurity, 'permutation_AP_drop': perm_mean, 'permutation_std': perm_std,
}).sort_values('impurity', ascending=False)
importance_table['group'] = ['raw' if name in RC_RAW_COLS else 'embedding'
                             for name in importance_table.index]
display(importance_table.head(15).round(4))
importance_table.to_csv(RC_OUT / 'tables' / 'RF2_feature_importance.csv')

rf1_importance = pd.Series(rf1_tuned.feature_importances_, index=list(rf1_tuned.feature_names_in_))
display(pd.DataFrame({
    'RF1_tuned': rf1_importance[RC_RAW_COLS],
    'RF2_tuned (same raw columns)': impurity[RC_RAW_COLS],
}).round(4))

top_cols = importance_table.head(15).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top_cols.index, top_cols['impurity'],
         color=['tab:orange' if g == 'embedding' else 'tab:blue' for g in top_cols['group']])
plt.xlabel('Impurity importance (blue = raw feature, orange = GCN embedding)')
plt.title('RF2 tuned: the 15 most used columns')
rc_save_figure('RF2_feature_importance')

### 20.3 Ablation: replace the graph operator `P` by the identity matrix `I`

With `P = I` every wallet only sees **itself** (the self-loop weight is 1), so the GCN becomes an ordinary two-layer network on the 12 features and no edge is used. Everything else is kept the same as in Section 13: architecture, optimiser, learning rate, weight decay, 200 epochs, patience 30 and checkpoint selection by validation AP. RF2 is then refitted on `[12 raw features | 32 embedding columns]` with the **same forest settings** (the initial configuration and the tuned configuration of Section 15).

How to read the result (validation wallets only, mean and standard deviation over GCN seeds; the forest seed stays 42):

- RF2 with `I` about equal to RF2 with `P`: the gain over RF1 does not come from the edges.
- RF2 with `P` above RF2 with `I` by more than the seed-to-seed spread: the edges add information.
- Both about equal to RF1: the embeddings add little at all.

The tuned forest settings were selected for the `P` embeddings, so they slightly favour `P`; the initial configuration has no such bias. The test set is not touched.

In [ ]:
def abl_identity_operator():
    """Sparse identity: every wallet keeps only its own representation."""
    idx = torch.arange(GCN_N, device=gcn_device)
    return torch.sparse_coo_tensor(
        torch.stack([idx, idx]),
        torch.ones(GCN_N, dtype=torch.float32, device=gcn_device),
        (GCN_N, GCN_N),
    ).coalesce()


abl_operators = {'P (graph)': gcn_adj, 'I (no edges)': abl_identity_operator()}


def abl_train_gcn(adjacency, seed):
    """Same training recipe as Section 13.6, for any operator and seed.

    Returns the embeddings of the best-validation-AP checkpoint, its validation AP and epoch.
    """
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = WalletGCN(**gcn_config).to(gcn_device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=GCN_LR, weight_decay=GCN_WEIGHT_DECAY)
    best_ap, best_epoch, best_state, stale = -float('inf'), 0, None, 0
    for epoch in range(1, GCN_MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(gcn_x, adjacency)
        loss = criterion(logits[gcn_train_idx], gcn_y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            eval_logits, _ = model(gcn_x, adjacency)
            val_scores = eval_logits[gcn_val_idx].softmax(dim=1)[:, 1].cpu().numpy()
        val_ap = float(metrics.average_precision_score(gcn_y_val_np, val_scores))
        if val_ap > best_ap + 1e-6:
            best_ap, best_epoch, stale = val_ap, epoch, 0
            best_state = {name: tensor.detach().cpu().clone()
                          for name, tensor in model.state_dict().items()}
        else:
            stale += 1
            if stale >= GCN_PATIENCE:
                break
    if best_state is None:
        raise RuntimeError('No valid checkpoint was selected.')
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        embeddings = model.encode(gcn_x, adjacency).cpu().numpy()
    return embeddings, best_ap, best_epoch

In [ ]:
RUN_ABLATION = True  # set to False to skip retraining the GCN 10 times
ABL_SEEDS = [42, 43, 44, 45, 46]  # use [42, 43, 44] for a quicker first run
ABL_RF_CONFIGS = {'initial': RC_INITIAL, 'tuned': rf2_best_params}


def abl_hybrid_inputs(embeddings):
    """Join the raw features and the given embeddings by wallet ID (train and validation only)."""
    emb = pd.DataFrame(embeddings, index=gcn_node_ids, columns=RC_H_COLS)
    joined = {}
    for split in ['train', 'val']:
        nodes = rc_frames[split]['node']
        joined[split] = pd.concat(
            [rc_inputs['RF1'][split].reset_index(drop=True),
             emb.loc[nodes].reset_index(drop=True)], axis=1,
        )[RC_HYBRID_COLS]
    return joined


if RUN_ABLATION:
    abl_rows = []
    for seed in ABL_SEEDS:
        for operator_name, operator in abl_operators.items():
            start = time.perf_counter()
            embeddings, gcn_ap, gcn_epoch = abl_train_gcn(operator, seed)
            if seed == GCN_SEED and operator_name.startswith('P'):
                print('Reproduction check (seed 42, P): max |H - gcn_H| =',
                      float(np.abs(embeddings - gcn_H).max()))
            joined = abl_hybrid_inputs(embeddings)
            for config_name, config in ABL_RF_CONFIGS.items():
                forest = RandomForestClassifier(**RC_FIXED_RF, **config)
                forest.fit(joined['train'], rc_y_train)
                result = rc_metrics(rc_y_val, rc_scores(forest, joined['val']))
                abl_rows.append({
                    'seed': seed, 'operator': operator_name, 'rf_config': config_name,
                    'val_AP': result['AP'], 'val_ROC_AUC': result['ROC_AUC'],
                    'gcn_head_val_AP': gcn_ap, 'gcn_best_epoch': gcn_epoch,
                })
            print(f'seed {seed} | {operator_name:12s} | GCN head val AP {gcn_ap:.4f} '
                  f'(epoch {gcn_epoch}) | {time.perf_counter() - start:.1f} s')

    abl = pd.DataFrame(abl_rows)
    abl.to_csv(RC_OUT / 'tables' / 'ablation_identity_operator.csv', index=False)

    rf1_reference = {
        'initial': rc_metrics(rc_y_val, rc_scores(rf1_initial, rc_inputs['RF1']['val'])),
        'tuned': rc_metrics(rc_y_val, rc_scores(rf1_tuned, rc_inputs['RF1']['val'])),
    }
    print('RF1 reference on validation (AP):',
          {name: round(result['AP'], 4) for name, result in rf1_reference.items()})

    summary = abl.groupby(['rf_config', 'operator']).agg(
        val_AP_mean=('val_AP', 'mean'), val_AP_std=('val_AP', 'std'),
        val_AUC_mean=('val_ROC_AUC', 'mean'), gcn_head_AP_mean=('gcn_head_val_AP', 'mean'),
    ).round(4)
    display(summary)

    paired = abl.pivot_table(index=['rf_config', 'seed'], columns='operator', values='val_AP')
    paired['P minus I'] = paired['P (graph)'] - paired['I (no edges)']
    display(paired.round(4))
    for config_name in ABL_RF_CONFIGS:
        diff = paired.loc[config_name, 'P minus I']
        print(f'{config_name}: mean(P - I) = {diff.mean():+.4f}, std = {diff.std():.4f}, '
              f'P better in {int((diff > 0).sum())} of {len(diff)} seeds')

    labels, means, spreads = [], [], []
    for config_name in ABL_RF_CONFIGS:
        for operator_name in abl_operators:
            ap = abl.loc[(abl['rf_config'] == config_name) & (abl['operator'] == operator_name), 'val_AP']
            labels.append(f'{config_name}\n{operator_name}')
            means.append(ap.mean())
            spreads.append(ap.std() if len(ap) > 1 else 0.0)
    low = min(means + [r['AP'] for r in rf1_reference.values()]) - 0.03
    plt.figure(figsize=(8, 4.8))
    plt.bar(range(len(labels)), means, yerr=spreads, capsize=4, color='tab:blue', alpha=0.8)
    plt.axhline(rf1_reference['initial']['AP'], linestyle='--', color='tab:gray', label='RF1 initial')
    plt.axhline(rf1_reference['tuned']['AP'], linestyle=':', color='tab:red', label='RF1 tuned')
    plt.xticks(range(len(labels)), labels, fontsize=8)
    plt.ylim(low, 1.0)
    plt.ylabel('Validation AP (mean ± std over GCN seeds)')
    plt.title('RF2 with the graph operator P versus the identity I')
    plt.legend()
    rc_save_figure('ablation_identity_operator')
else:
    print('Ablation skipped (RUN_ABLATION = False).')

### 20.4 Rule-based baseline: one raw feature as the score

A hand-written rule such as "flag wallets with many incoming transfers" has no learned parameters. To measure how much machine learning adds over such a rule, each of the 12 raw features is used directly as a ranking score, in both directions (higher or lower = more suspicious). The feature and direction with the best **training** AP are chosen, and only that rule is scored on the **validation** wallets and compared with RF1 and RF2. The test set is not touched, so the frozen test protocol of Section 17 is unchanged.

In [ ]:
rule_rows = []
for rule_feature in RC_RAW_COLS:
    rule_train_values = rc_inputs['RF1']['train'][rule_feature].to_numpy(dtype=float)
    for rule_direction, rule_sign in [('higher is more suspicious', 1.0), ('lower is more suspicious', -1.0)]:
        rule_scores = rule_sign * rule_train_values
        rule_rows.append({
            'feature': rule_feature, 'direction': rule_direction, 'sign': rule_sign,
            'train_AP': float(metrics.average_precision_score(rc_y_train, rule_scores)),
            'train_ROC_AUC': float(metrics.roc_auc_score(rc_y_train, rule_scores)),
        })
rule_table = pd.DataFrame(rule_rows).sort_values('train_AP', ascending=False).reset_index(drop=True)
print('Best rules on the TRAINING wallets (selection uses train only):')
display(rule_table.head(6).round(4))
rule_table.to_csv(RC_OUT / 'tables' / 'rule_baseline_candidates_train.csv', index=False)

rule_choice = rule_table.iloc[0]
rule_val_scores = rule_choice['sign'] * rc_inputs['RF1']['val'][rule_choice['feature']].to_numpy(dtype=float)
rule_label = f"Rule: {rule_choice['feature']} ({rule_choice['direction']})"
print('Chosen rule:', rule_label)

rule_comparison = [
    {'ranking': 'No skill (positive fraction)', 'val_AP': float(np.mean(rc_y_val)), 'val_ROC_AUC': 0.5},
    {'ranking': rule_label,
     'val_AP': float(metrics.average_precision_score(rc_y_val, rule_val_scores)),
     'val_ROC_AUC': float(metrics.roc_auc_score(rc_y_val, rule_val_scores))},
]
for model_name in ['RF1_initial', 'RF1_tuned', 'RF2_initial', 'RF2_tuned']:
    rule_comparison.append({
        'ranking': model_name,
        'val_AP': float(metrics.average_precision_score(rc_y_val, rc_val_scores[model_name])),
        'val_ROC_AUC': float(metrics.roc_auc_score(rc_y_val, rc_val_scores[model_name])),
    })
rule_comparison = pd.DataFrame(rule_comparison)
print('Validation wallets only (the test set is not used):')
display(rule_comparison.round(4))
rule_comparison.to_csv(RC_OUT / 'tables' / 'rule_baseline_validation.csv', index=False)

rc_plot_pr(
    rc_y_val,
    {rule_label: rule_val_scores,
     'RF1_tuned': rc_val_scores['RF1_tuned'],
     'RF2_tuned': rc_val_scores['RF2_tuned']},
    'Validation', 'rule_baseline',
)

## 21. Report-ready summary and backup

The code below fills a short summary **from the results computed in this run**. It does not prefill a claim that RF2 wins. A lower score for the hybrid would be a valid outcome to discuss, not a reason to retune on the test set.

**Limitations of this experiment:**

- This remains a retrospective, transductive EPTransNet benchmark, sampled from a crawl started around reported phishing accounts. Train/validation/test positive rates (50%, 16.67%, 4.76%) are experimental. `isp=0` is not independently verified safe.
- Account features use the available full released history, not necessarily all on-chain transactions or data available before an account was reported. This does not demonstrate prospective early detection.
- The preprocessing above keeps its historical fallback conversion (a non-convertible amount becomes 0, a non-convertible timestamp becomes `None`), which can conceal malformed values; an upstream validation audit would be needed to resolve that limitation. Redundant derived features were omitted by design, not because they can never help a tree.
- The GCN was trained with training labels and checkpoint-selected on validation; frozen embeddings are not out-of-fold training embeddings. High hybrid train scores may be optimistic. Conventional cross-validation on these embeddings would not be end-to-end leakage-free.
- Both RFs get the same RF search grid, but the hybrid has additional GCN training and checkpoint selection. The primary result measures the value of the full hybrid pipeline, not a causal proof that edges alone produce any improvement.
- F1/precision/AP from different class prevalences are not directly comparable as pure evidence of overfitting. The validation-selected F1 threshold may transfer poorly to the rarer-positive test set.
- A single split/seed and these descriptive differences do not establish statistical significance or optimal performance. Any change guided by the test results would be exploratory and would need a fresh holdout for a new final claim.

In [ ]:
summary_lines = [
    '# RF1 versus GCN + RF2 — experiment summary',
    '',
    'The original preprocessing, sampling, feature definitions and GCN checkpoint were retained.',
    f'RF1 uses {len(RC_RAW_COLS)} raw features; RF2 uses those features plus {len(RC_H_COLS)} GCN embeddings.',
    f'Each RF was selected independently from {len(RC_CANDIDATES)} candidates by validation AP.',
    'The GCN was not retrained during RF tuning. No train+validation refit was performed.',
    '',
    '## Selected RF configurations',
    '```json', json.dumps(rc_best_params, indent=2), '```',
    '',
    '## Secondary validation-selected thresholds',
    '```json', json.dumps(rc_thresholds, indent=2), '```',
]

if rc_test_results is not None:
    fixed = rc_test_results.query("threshold_policy == 'fixed_0.5'").set_index('model')
    one, two = fixed.loc['RF1_tuned'], fixed.loc['RF2_tuned']
    delta_ap = float(two['AP'] - one['AP'])
    delta_f1 = float(two['F1_phishing'] - one['F1_phishing'])
    summary_lines += [
        '', '## Held-out test results',
        f'Test accounts: {len(rc_y_test)}; phishing-labelled: {int(rc_y_test.sum())}; '
        f'positive fraction: {rc_y_test.mean():.4f}.',
        f'Tuned RF1 test AP: {one["AP"]:.4f}. Tuned RF2 test AP: {two["AP"]:.4f}.',
        f'RF2 minus RF1 AP: {delta_ap:+.4f} (descriptive, not a significance claim).',
        f'At the common threshold 0.5, phishing F1 was {one["F1_phishing"]:.4f} '
        f'for RF1 and {two["F1_phishing"]:.4f} for RF2 (difference {delta_f1:+.4f}).',
        f'RF1: TP={int(one["TP"])}, FP={int(one["FP"])}, FN={int(one["FN"])}. '
        f'RF2: TP={int(two["TP"])}, FP={int(two["FP"])}, FN={int(two["FN"])}.',
    ]
    if delta_ap > 1e-12:
        summary_lines.append('The hybrid achieved higher test AP in this fixed experiment; this does not establish a general advantage.')
    elif delta_ap < -1e-12:
        summary_lines.append('The hybrid did not improve test AP over the tuned tabular baseline in this fixed experiment.')
    else:
        summary_lines.append('The tuned systems tied in test AP at the reported precision.')
    secondary = rc_test_results.query("threshold_policy == 'validation_F1'").set_index('model')
    summary_lines += ['', '## Secondary operating points (not selected on test)']
    for name in ['RF1_tuned', 'RF2_tuned']:
        row = secondary.loc[name]
        summary_lines.append(
            f'{name}: threshold={row["threshold"]:.4f}, precision={row["precision"]:.4f}, '
            f'recall={row["recall"]:.4f}, phishing F1={row["F1_phishing"]:.4f}.'
        )
else:
    summary_lines += ['', 'The test set has not been evaluated. No test comparison is claimed.']

summary_lines += [
    '', '## Interpretation limits',
    'These results concern the released labels and selected graph, not independently confirmed fraud or Ethereum-wide prevalence.',
    'The experiment is retrospective and transductive. Only the forest search budget is matched; the hybrid also required GCN training.',
    'Validation results were used for selection. The single test split and seed do not establish statistical significance.',
    'No unseen-data superiority or calibrated probability claim is made.',
]
summary_text = '\n'.join(summary_lines) + '\n'
(RC_OUT / 'comparison_summary.md').write_text(summary_text, encoding='utf-8')
print(summary_text)

# Include small GCN metadata needed to trace the embeddings, without copying the large graph.
lineage_dir = RC_OUT / 'gcn_lineage'
lineage_dir.mkdir(exist_ok=True)
for filename in ['gcn_config.json', 'preprocessing.json', 'gcn_best_state.pt']:
    source = RC_DATA_DIR / 'gcn_outputs' / filename
    if source.is_file():
        shutil.copy2(source, lineage_dir / filename)

archive = shutil.make_archive(str(RC_DATA_DIR / 'A2_comparison_results'), 'zip', root_dir=RC_OUT)
print('Results archive:', archive)
print('Save this ZIP AND your notebook. Runtime files are not stored inside the notebook.')
print('For full reproducibility also retain the input CSVs and gcn_outputs folder.')

# Optional Colab download, after checking the outputs:
# from google.colab import files
# files.download(archive)